# LifeMate — Làn B: agent eval / public submission

ASR transcript → prompt (ngữ cảnh, lịch sử kèm `tool_results`, schema các app đã cài) → **Qwen3-4B-Instruct-2507** sinh một JSON action mỗi bước → `env.validate` → `env.call` → … Mỗi mẫu đúng **một** `ToolEnv`; `tool_calls` và `state_hash` lấy thẳng từ môi trường, không bịa.

`CONFIG["mode"]`:
- `public` (mặc định): toàn bộ `public_test` → `/kaggle/working/submission.jsonl` (đã chạy validator `zaic_tools`). Nộp file này lên Public Leaderboard.
- `diagnostic`: train với transcript sạch (`audio_text`), tách lỗi planner khỏi lỗi ASR. Đặt `split="train"`, `subset="dev_seen"` (hoặc `dev_unseen`, `final_holdout`).
- `e2e`: train với transcript ASR (cache Làn A hoặc ASR chạy ngay trong notebook).

Transcript: notebook tìm cache ASR trong `/kaggle/input` (output của `lifemate_asr`) rồi trong output này; thiếu audio nào thì (nếu `asr_inline`) tự ASR phần thiếu bằng `asr_backend` trước khi chạy agent. Bật GPU T4 x2 + Internet, add dataset `thanhkhil/zaloai-lifemate-dataset`.

`CONFIG_PRESET` (cuối cell config) chọn một trong ba preset, mỗi version Kaggle chạy một preset:
- `public`: như trên, ASR lấy từ cache (dataset `lifemate-asr-cache` nếu đã Add Input).
- `dev`: `e2e` trên train `dev_seen+dev_unseen`, chấm điểm local (`merge_report.json`).
- `sft`: ASR toàn bộ train → LoRA trên `train_fit`, hai biến thể song song mỗi GPU (transcript gold / ASR) → merge vào base trong `/tmp` → `e2e` dev từng biến thể (`eval_<biến thể>/merge_report.json`, `sft_eval_summary.json`). Chỉ adapter (checkpoint) được lưu vào output.

Dùng adapter đã train cho `public`/`dev`: Add Input output của version `sft`, đặt `CONFIG["llm"]["adapter"]` = thư mục chứa `ckpt-*` của biến thể (vd `/kaggle/input/<kernel>/lm_agent/sft_asr`).

Hết giờ giữa chừng: chạy version mới, attach output cũ làm input, ghi đường dẫn vào `resume_from_dirs` (cùng config). Mẫu chưa xong **không** được điền hash giả: file nộp chỉ được ghi khi đủ mọi id.

In [ ]:
CONFIG = {
  "track": "lifemate",
  "experiment": "lm-B-005-budgetfix-public",
  "parent_run": None,
  "mode": "public",
  "split": "public_test",
  "subset": "all",
  "manifest_seed": 2026,
  "limit": None,
  "input_roots": [
    "/kaggle/input"
  ],
  "out_dir": "/kaggle/working/lm_agent",
  "asr_backend": "phowhisper",
  "asr_inline": True,
  "asr_cache_dirs": [
    "/kaggle/input",
    "/kaggle/working/lm_agent"
  ],
  "llm": {
    "backend": "auto",
    "model": "Qwen/Qwen3-4B-Instruct-2507",
    "revision": None,
    "dtype": "half",
    "max_model_len": 16384,
    "gpu_memory_utilization": 0.9,
    "enable_thinking": False,
    "load_in_4bit": False,
    "batch_size": 8,
    "seed": 0,
    "adapter": None
  },
  "agent": {
    "max_steps": 8,
    "max_new_tokens": 256,
    "max_invalid": 2,
    "sample_budget_s": 120.0,
    "result_chars": 1500
  },
  "batch_size": 32,
  "cpu_threads": 1,
  "seed": 0,
  "wallclock_s": 37800.0,
  "install_vllm": True,
  "resume_from_dirs": []
}
ASR_BACKENDS = {
  "phowhisper": {
    "backend": "phowhisper",
    "model": "vinai/PhoWhisper-medium",
    "revision": None,
    "batch_size": 16
  },
  "gipformer": {
    "backend": "gipformer",
    "model": "g-group-ai-lab/gipformer-68M-rnnt",
    "revision": None,
    "int8": False,
    "num_threads": 2,
    "batch_size": 32
  }
}
# LoRA SFT (mode "sft"): one training process per GPU, one variant each; then e2e dev eval of every variant
SFT = {
  "train_subset": "train_fit",
  "variants": ["gold", "asr"],  # transcripts used in the training prompts
  "lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.05,
  "lr": 1e-4, "warmup_ratio": 0.03, "epochs": 1, "grad_accum": 16, "max_len": 8192,
  "gradient_checkpointing": True, "seed": 0,
  "ckpt_every_steps": 25, "ckpt_every_min": 20,
  "wallclock_s": 7 * 3600,  # ~3k examples x ~4k tokens on one T4; stops with a checkpoint
}

# One notebook source, one preset per Kaggle version (overrides on top of CONFIG)
CONFIG_PRESET = "dev_qwen35"  # public | dev | sft | dev_qwen35 | dev_gipformer
PRESETS = {
  "public": {},
  "dev": {"experiment": "lm-B-006-budgetfix-dev", "mode": "e2e", "split": "train", "subset": "dev_seen+dev_unseen"},
  "sft": {"experiment": "lm-sft-007-lora-gold-vs-asr", "mode": "sft", "split": "train", "subset": "dev_seen+dev_unseen"},
  # same as "dev", one factor changed each (compare with lm-B-006)
  "dev_qwen35": {"experiment": "lm-B-008-qwen35-4b-dev", "mode": "e2e", "split": "train", "subset": "dev_seen+dev_unseen",
                 "llm": dict(CONFIG["llm"], model="Qwen/Qwen3.5-4B")},
  "dev_gipformer": {"experiment": "lm-A-009-gipformer-dev", "mode": "e2e", "split": "train", "subset": "dev_seen+dev_unseen",
                    "asr_backend": "gipformer"},
}
CONFIG.update(PRESETS[CONFIG_PRESET])
print(CONFIG_PRESET, CONFIG["experiment"], CONFIG["mode"], CONFIG["split"], CONFIG["subset"])

In [ ]:
import os, sys, json, time, glob, shutil, subprocess
from pathlib import Path
WORK = Path("/kaggle/working")
LIB = WORK / "lib"
(LIB / "lmlib").mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(LIB))
OUT = Path(CONFIG["out_dir"]); OUT.mkdir(parents=True, exist_ok=True)
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"  # zaic_tools lives on the read-only input

def sh(cmd):
    print("$", cmd); r = subprocess.run(cmd, shell=True, capture_output=True, text=True); print((r.stdout + r.stderr)[-3000:]); return r.returncode

## Thư viện `lmlib` (nhúng sẵn)

In [ ]:
%%writefile /kaggle/working/lib/lmlib/__init__.py
"""LifeMate library embedded into the Kaggle notebooks (see build_notebooks.py)."""

In [ ]:
%%writefile /kaggle/working/lib/lmlib/common.py
"""Run plumbing shared by the LifeMate notebooks: durable JSONL shards with resume, stable sharding, config hashes,
atomic writes, input discovery under /kaggle/input and environment info."""
from __future__ import annotations

import hashlib
import json
import os
import platform
import subprocess
import time
from pathlib import Path


def dumps(obj) -> str:
    return json.dumps(obj, ensure_ascii=False, separators=(",", ":"), allow_nan=False, default=str)


def load_jsonl(path) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def read_jsonl_tolerant(path) -> tuple[list[dict], int]:
    """Rows of a JSONL file written by a process that may have died mid-line. A broken *last* line is dropped
    (returns how many were dropped); a broken line in the middle means the file is corrupt -> ValueError."""
    rows, lines = [], Path(path).read_text(encoding="utf-8").split("\n")
    while lines and not lines[-1].strip():
        lines.pop()
    dropped = 0
    for i, line in enumerate(lines):
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError:
            if i == len(lines) - 1:
                dropped = 1
            else:
                raise ValueError(f"{path}: corrupt line {i + 1}")
    return rows, dropped


def atomic_write_text(path, text: str) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(text)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)


def atomic_write_json(path, obj) -> None:
    atomic_write_text(path, json.dumps(obj, ensure_ascii=False, indent=1, default=str))


def sha256_file(path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def config_hash(cfg: dict, ignore=()) -> str:
    """Hash of the settings that change results (shard/output/resume keys go in `ignore`)."""
    keep = {k: v for k, v in cfg.items() if k not in ignore}
    return hashlib.sha256(json.dumps(keep, sort_keys=True, default=str).encode()).hexdigest()[:16]


def shard_of(key: str, num_shards: int) -> int:
    """Stable shard index (does not depend on PYTHONHASHSEED or on the order of the inputs)."""
    if num_shards <= 1:
        return 0
    return int(hashlib.md5(key.encode()).hexdigest(), 16) % num_shards


class DurableJsonl:
    """Append-only result file of one worker. First line is a header with the run hash; every record is flushed and
    fsync'ed. On reopen: a truncated last line is dropped, the file is rewritten clean, and a header with another run
    hash is refused (never mix results of different configs). done_ids() = ids whose latest record is 'completed'."""

    def __init__(self, path, run_hash: str, metadata: dict | None = None):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.run_hash = run_hash
        self.records: dict[str, dict] = {}
        if self.path.exists() and self.path.stat().st_size:
            rows, dropped = read_jsonl_tolerant(self.path)
            header = rows[0].get("_header") if rows else None
            if not header or header.get("run_hash") != run_hash:
                raise RuntimeError(f"{self.path} belongs to run {header and header.get('run_hash')}, not {run_hash}")
            for r in rows[1:]:
                self.records[r["id"]] = r
            if dropped:
                atomic_write_text(self.path, "\n".join(dumps(r) for r in rows) + "\n")
        else:
            atomic_write_text(self.path, dumps({"_header": {"run_hash": run_hash, "created": time.time(), **(metadata or {})}}) + "\n")
        self._f = open(self.path, "a", encoding="utf-8")

    def done_ids(self) -> set[str]:
        return {i for i, r in self.records.items() if r.get("status") == "completed"}

    def append(self, rec: dict) -> None:
        previous = self.records.get(rec["id"])
        if previous and previous.get("status") == "completed":
            if previous != rec:
                raise ValueError(f"conflicting completed record: {rec['id']}")
            return
        self._f.write(dumps(rec) + "\n")
        self._f.flush()
        os.fsync(self._f.fileno())
        self.records[rec["id"]] = rec

    def close(self) -> None:
        if not self._f.closed:
            self._f.close()


def find_split_dir(split: str, roots=("/kaggle/input",), marker: str = "public_test.jsonl", max_depth: int = 6) -> Path:
    """Directory `<...>/<split>` that holds `marker` (datasets get mounted as /kaggle/input/<slug>/<split>/<split>/...)."""
    for root in map(Path, roots):
        if not root.exists():
            continue
        base_depth = len(root.parts)
        for dirpath, dirnames, filenames in os.walk(root):
            p = Path(dirpath)
            if len(p.parts) - base_depth > max_depth:
                dirnames[:] = []
                continue
            if p.name == split and marker in filenames:
                return p
    raise FileNotFoundError(f"no {split}/{marker} under {list(roots)}")


def env_info() -> dict:
    info = {"python": platform.python_version(), "platform": platform.platform(), "cpus": os.cpu_count()}
    try:
        info["gpus"] = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True, timeout=20
        ).stdout.strip().splitlines()
    except Exception:
        info["gpus"] = []
    for mod in ("torch", "transformers", "vllm", "pandas"):
        try:
            info[mod] = __import__(mod).__version__
        except Exception:
            pass
    try:
        with open("/proc/meminfo") as f:
            info["mem_total_kb"] = int(f.readline().split()[1])
    except Exception:
        pass
    return info


def set_thread_env(threads: int = 1) -> None:
    """One BLAS/OpenMP thread per process (Kaggle has few CPUs)."""
    for k in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        os.environ.setdefault(k, str(threads))


class Deadline:
    """Wall-clock budget of a session: stop taking new work at `stop_after_s`."""

    def __init__(self, stop_after_s: float, end_epoch=None, reserve_s=0):
        self.t0 = time.time()
        self.stop_after_s = min(stop_after_s, end_epoch - self.t0 if end_epoch is not None else stop_after_s) - reserve_s

    def elapsed(self) -> float:
        return time.time() - self.t0

    def expired(self) -> bool:
        return self.elapsed() >= self.stop_after_s

def session_deadline(cfg):
    return Deadline(float(cfg.get("session_wallclock_s", cfg.get("wallclock_s", 14400))),
                    cfg.get("session_deadline_epoch"), cfg.get("export_reserve_s", 0))


def can_start(deadline, estimate_s=0):
    return deadline.elapsed() + max(float(estimate_s), 0) < deadline.stop_after_s


class MemoryMonitor:
    """Sample this worker's nvidia-smi memory, including backend subprocesses on its visible GPU."""
    def __init__(self):
        import threading
        self.peak_mb = None
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()

    def _poll(self):
        while not self._stop.is_set():
            try:
                raw = subprocess.run(["nvidia-smi", "--query-compute-apps=pid,used_memory", "--format=csv,noheader,nounits"],
                                     capture_output=True, text=True, timeout=3).stdout
                # Main worker + descendants (vLLM engine processes), excluding unrelated workers.
                parents = {os.getpid()}
                proc_parents = {}
                for p in Path("/proc").glob("[0-9]*/status"):
                    try:
                        fields = dict(l.split(":", 1) for l in p.read_text().splitlines() if ":" in l)
                        proc_parents[int(p.parent.name)] = int(fields["PPid"].strip())
                    except (OSError, ValueError, KeyError): pass
                while True:
                    new = {pid for pid, parent in proc_parents.items() if parent in parents} - parents
                    if not new: break
                    parents.update(new)
                used = sum(int(line.split(",")[1]) for line in raw.splitlines()
                           if line.strip() and int(line.split(",")[0]) in parents)
                self.peak_mb = max(self.peak_mb or 0, used)
            except (OSError, ValueError, subprocess.TimeoutExpired): pass
            self._stop.wait(.5)

    def close(self):
        self._stop.set()
        self._thread.join(timeout=4)
        return self.peak_mb


In [ ]:
%%writefile /kaggle/working/lib/lmlib/llm.py
"""Greedy chat generation behind one interface: vLLM (fast, Kaggle T4 in fp16), HF transformers (fallback) or a mock.

Prompts are rendered with the model's own chat template (enable_thinking passed when the template knows it) and
prompts that do not fit `max_model_len - max_tokens` are not generated but returned with finish='too_long'.
"""
from __future__ import annotations

import os
import random
from dataclasses import dataclass
from typing import Callable


@dataclass
class GenOut:
    text: str
    prompt_tokens: int
    output_tokens: int
    finish: str  # stop | length | too_long | error


def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import numpy as np

        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


class BaseLLM:
    max_model_len: int = 16384

    def render(self, conv: list[dict]) -> str:
        raise NotImplementedError

    def count_tokens(self, conv: list[dict]) -> int:
        raise NotImplementedError

    def generate(self, convs: list[list[dict]], max_tokens: int) -> list[GenOut]:
        raise NotImplementedError


def _apply_template(tok, conv, thinking: bool) -> str:
    try:
        return tok.apply_chat_template(conv, tokenize=False, add_generation_prompt=True, enable_thinking=thinking)
    except TypeError:
        return tok.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)


class VLLM(BaseLLM):
    def __init__(self, model: str, revision: str | None = None, dtype: str = "half", max_model_len: int = 16384,
                 gpu_memory_utilization: float = 0.90, quantization: str | None = None, seed: int = 0,
                 enable_thinking: bool = False, tensor_parallel_size: int = 1, enforce_eager: bool = False, **_):
        from vllm import LLM, SamplingParams  # noqa: F401

        self._SamplingParams = SamplingParams
        self.llm = LLM(model=model, revision=revision, dtype=dtype, max_model_len=max_model_len, seed=seed,
                       gpu_memory_utilization=gpu_memory_utilization, quantization=quantization,
                       tensor_parallel_size=tensor_parallel_size, enable_prefix_caching=True, enforce_eager=enforce_eager)
        self.tok = self.llm.get_tokenizer()
        self.max_model_len = max_model_len
        self.thinking = enable_thinking

    def render(self, conv):
        return _apply_template(self.tok, conv, self.thinking)

    def count_tokens(self, conv):
        return len(self.tok(self.render(conv), add_special_tokens=False)["input_ids"])

    def generate(self, convs, max_tokens):
        prompts = [self.render(c) for c in convs]
        lens = [len(self.tok(p, add_special_tokens=False)["input_ids"]) for p in prompts]
        todo = [i for i, n in enumerate(lens) if n + max_tokens <= self.max_model_len]
        outs = [GenOut("", n, 0, "too_long") for n in lens]
        if todo:
            sp = self._SamplingParams(temperature=0.0, top_p=1.0, max_tokens=max_tokens, seed=0)
            res = self.llm.generate([prompts[i] for i in todo], sp, use_tqdm=False)
            for i, r in zip(todo, res):
                o = r.outputs[0]
                outs[i] = GenOut(o.text, lens[i], len(o.token_ids), "length" if o.finish_reason == "length" else "stop")
        return outs


class HFLLM(BaseLLM):
    def __init__(self, model: str, revision: str | None = None, dtype: str = "half", max_model_len: int = 16384,
                 load_in_4bit: bool = False, enable_thinking: bool = False, batch_size: int = 4, device_map="auto", **_):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.torch = torch
        self.tok = AutoTokenizer.from_pretrained(model, revision=revision)
        self.tok.padding_side = "left"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        kw = {"revision": revision, "device_map": device_map}
        if load_in_4bit:
            from transformers import BitsAndBytesConfig

            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
        else:
            kw["torch_dtype"] = torch.float16 if dtype in ("half", "float16") else (torch.bfloat16 if dtype == "bfloat16" else torch.float32)
        self.model = AutoModelForCausalLM.from_pretrained(model, **kw).eval()
        self.max_model_len = max_model_len
        self.thinking = enable_thinking
        self.batch_size = batch_size

    def render(self, conv):
        return _apply_template(self.tok, conv, self.thinking)

    def count_tokens(self, conv):
        return len(self.tok(self.render(conv), add_special_tokens=False)["input_ids"])

    def generate(self, convs, max_tokens):
        prompts = [self.render(c) for c in convs]
        lens = [len(self.tok(p, add_special_tokens=False)["input_ids"]) for p in prompts]
        outs = [GenOut("", n, 0, "too_long") for n in lens]
        todo = [i for i, n in enumerate(lens) if n + max_tokens <= self.max_model_len]
        todo.sort(key=lambda i: lens[i])
        for b in range(0, len(todo), self.batch_size):
            idx = todo[b : b + self.batch_size]
            enc = self.tok([prompts[i] for i in idx], return_tensors="pt", padding=True, add_special_tokens=False).to(self.model.device)
            with self.torch.no_grad():
                gen = self.model.generate(**enc, max_new_tokens=max_tokens, do_sample=False, temperature=None, top_p=None,
                                          top_k=None, pad_token_id=self.tok.pad_token_id)
            for row, i in zip(gen, idx):
                new = row[enc["input_ids"].shape[1] :]
                n_out = int((new != self.tok.pad_token_id).sum())
                text = self.tok.decode(new, skip_special_tokens=True)
                outs[i] = GenOut(text, lens[i], n_out, "length" if n_out >= max_tokens else "stop")
        return outs


class MockLLM(BaseLLM):
    """Deterministic stand-in for tests and CPU dry-runs: fn(conv) -> text."""

    def __init__(self, fn: Callable[[list[dict]], str], max_model_len: int = 16384, **_):
        self.fn = fn
        self.max_model_len = max_model_len

    def render(self, conv):
        return "\n".join(f"<{m['role']}>{m['content']}" for m in conv)

    def count_tokens(self, conv):
        return len(self.render(conv)) // 3

    def generate(self, convs, max_tokens):
        out = []
        for c in convs:
            n = self.count_tokens(c)
            out.append(GenOut("", n, 0, "too_long") if n + max_tokens > self.max_model_len else GenOut(self.fn(c), n, 1, "stop"))
        return out


def make_llm(cfg: dict, mock_fn: Callable | None = None) -> BaseLLM:
    """cfg: backend (auto|vllm|hf|mock), model, revision, dtype, max_model_len, gpu_memory_utilization, quantization,
    load_in_4bit, enable_thinking, seed."""
    backend = cfg.get("backend", "auto")
    seed_everything(int(cfg.get("seed", 0)))
    if backend == "mock":
        return MockLLM(mock_fn or (lambda conv: "{\"done\": true}"), **cfg)
    if backend in ("auto", "vllm"):
        try:
            return VLLM(**{k: v for k, v in cfg.items() if k != "backend"})
        except Exception as e:  # not installed, or no kernel for this GPU (e.g. T4): fall back to transformers
            if backend == "vllm":
                raise
            print(f"[llm] vLLM unavailable ({type(e).__name__}: {e}); using transformers", flush=True)
    return HFLLM(**{k: v for k, v in cfg.items() if k != "backend"})

In [ ]:
%%writefile /kaggle/working/lib/lmlib/data.py
"""LifeMate split loading, audio lists and the fixed train manifest."""
from __future__ import annotations

import hashlib
import random
import re
from collections import Counter
from pathlib import Path

from .common import find_split_dir, load_jsonl


def resolve_split_dir(cfg: dict) -> Path:
    if cfg.get("split_dir"):
        return Path(cfg["split_dir"])
    split = cfg["split"]
    return find_split_dir(split, cfg.get("input_roots", ["/kaggle/input"]), marker=f"{split}.jsonl")


def load_rows(split_dir: Path) -> list[dict]:
    return load_jsonl(split_dir / f"{split_dir.name}.jsonl")


def audio_paths(row: dict) -> list[str]:
    """Relative paths of every user audio of a sample: history turns first, then the final request."""
    out = [t["audio"] for t in row.get("history", []) if t.get("role") == "user" and t.get("audio")]
    return out + [row["audio"]]


def _template_key(row: dict) -> str:
    text = re.sub(r"\d+", "#", (row.get("audio_text") or "").lower())
    names = ",".join(sorted({c["name"] for c in (row.get("label") or {}).get("tool_calls", [])}))
    return hashlib.md5(f"{names}|{text[:40]}".encode()).hexdigest()


def label_apps(row: dict) -> set[str]:
    calls = list((row.get("label") or {}).get("tool_calls", []))
    for t in row.get("history", []):
        calls += t.get("tool_calls", []) or []
    return {c["name"].split(".", 1)[0] for c in calls if not c["name"].startswith("sys.")}


def make_manifest(rows: list[dict], seed: int = 2026, holdout_apps: int = 4, dev_frac: float = 0.12, final_frac: float = 0.12) -> dict:
    """train_fit / dev_seen / dev_unseen / final_holdout. dev_unseen = samples that use one of `holdout_apps` apps
    (drawn among apps used in 20..120 samples); no train_fit sample uses them. Near-duplicates (same tools + same
    text opening) stay in one part."""
    rng = random.Random(seed)
    use = Counter(a for r in rows for a in label_apps(r))
    cands = sorted(a for a, n in use.items() if 20 <= n <= 120)
    held = sorted(rng.sample(cands, min(holdout_apps, len(cands))))
    unseen = [r["id"] for r in rows if label_apps(r) & set(held)]
    rest = [r for r in rows if r["id"] not in set(unseen)]
    groups: dict[str, list[str]] = {}
    for r in rest:
        groups.setdefault(_template_key(r), []).append(r["id"])
    keys = sorted(groups)
    rng.shuffle(keys)
    n = len(rest)
    ids = {"train_fit": [], "dev_seen": [], "final_holdout": [], "dev_unseen": unseen}
    for k in keys:
        if len(ids["final_holdout"]) < final_frac * n:
            ids["final_holdout"] += groups[k]
        elif len(ids["dev_seen"]) < dev_frac * n:
            ids["dev_seen"] += groups[k]
        else:
            ids["train_fit"] += groups[k]
    return {"seed": seed, "holdout_apps": held, "ids": ids, "counts": {k: len(v) for k, v in ids.items()}}

def prepare_dev_run(cfg: dict, rows: list[dict]) -> dict:
    from .provenance import digest, package_hash
    ids = [r['id'] for r in rows]
    if len(ids) != len(set(ids)): raise ValueError('duplicate episode IDs')
    manifest = make_manifest(sorted(rows, key=lambda r:r['id']), seed=cfg.get('manifest_seed',2026))
    subset=cfg.get('subset','dev_seen')
    selected=sorted(set(subset) if isinstance(subset,list) else
                    (set(ids) if subset=='all' else set().union(*(manifest['ids'][s] for s in subset.split('+')))))
    if not set(selected)<=set(ids):raise ValueError('unknown episode ID')
    for i,a in enumerate(manifest['ids']):
        for b in list(manifest['ids'])[i+1:]:
            if set(manifest['ids'][a]) & set(manifest['ids'][b]):raise ValueError('overlapping split')
    manifest.update(selected_ids=selected, input_sha256=digest(sorted(rows,key=lambda r:r['id'])),
                    heldout_ids_sha256=digest(manifest['ids']['final_holdout']),
                    code_sha256=package_hash(Path(__file__).parent))
    manifest['manifest_sha256']=digest({'ids':manifest['ids'],'selected_ids':selected,'seed':manifest['seed']})
    return manifest


def prepare_audio_manifest(rows: list[dict], episode_ids: list[str], split_dir=None) -> dict:
    from .provenance import digest
    from .common import sha256_file
    wanted=set(episode_ids);selected={r['id']:r for r in rows if r['id'] in wanted}
    if set(selected)!=wanted:raise ValueError('missing episode ID')
    roles={};gold={};sources={}
    for qid in sorted(selected):
        r=selected[qid]
        turns=[(f'history:{i}',t) for i,t in enumerate(r.get('history',[])) if t.get('role')=='user']+[('final',r)]
        for role,t in turns:
            audio=t.get('audio')
            if not audio:raise ValueError(f'missing user audio: {qid}/{role}')
            roles.setdefault(audio,[]).append({'episode_id':qid,'role':role})
            if t.get('audio_text') is not None:
                if audio in gold and gold[audio]!=t['audio_text']:raise ValueError(f'conflicting gold transcript: {audio}')
                gold[audio]=t['audio_text']
    for audio in sorted(roles):
        if split_dir is None:raise ValueError('split_dir is required to fingerprint audio bytes')
        root=Path(split_dir).resolve();p=(root/audio).resolve()
        if not p.is_relative_to(root) or not p.is_file():raise ValueError(f'missing/unsafe audio: {audio}')
        duration=None
        try:
            import soundfile as sf
            duration=sf.info(p).duration
        except (ImportError, RuntimeError):pass
        sources[audio]={'sha256':sha256_file(p),'bytes':p.stat().st_size,'duration_s':duration}
    payload={'audio_ids':sorted(roles),'episode_roles':roles,'sources':sources}
    payload['audio_manifest_sha256']=digest(payload)
    return payload


In [ ]:
%%writefile /kaggle/working/lib/lmlib/asr.py
"""Lane A: transcribe every user audio of a split into a durable cache (one record per audio path), and measure
WER / CER / entity errors on train where audio_text exists.

    python -m lmlib.asr --config cfg.json --shard I      # worker; launch() starts one per GPU
Backends: phowhisper (transformers, fp16 GPU), gipformer (sherpa-onnx RNN-T, CPU), gold (train audio_text; dry-run).
"""
from __future__ import annotations

import argparse
import json
import os
import re
import subprocess
import sys
import time
import unicodedata
from pathlib import Path

from .common import DurableJsonl, Deadline, atomic_write_json, config_hash, env_info, read_jsonl_tolerant, set_thread_env, shard_of
from .data import audio_paths, load_rows, resolve_split_dir

RUN_IGNORE = ("shard_index", "num_shards", "out_dir", "resume", "wallclock_s", "input_roots", "split_dir", "limit", "subset_ids")


# ------------------------------------------------------------------ backends
class PhoWhisper:
    def __init__(self, model="vinai/PhoWhisper-medium", revision=None, batch_size=16, **_):
        import torch
        from transformers import pipeline

        self.pipe = pipeline("automatic-speech-recognition", model=model, revision=revision, torch_dtype=torch.float16,
                             device=0 if torch.cuda.is_available() else -1)
        self.batch_size = batch_size

    def transcribe(self, waves: list) -> list[str]:
        inputs = [{"raw": w, "sampling_rate": 16000} for w in waves]
        out = self.pipe(inputs, batch_size=self.batch_size, generate_kwargs={"do_sample": False, "num_beams": 1})
        return [o["text"].strip() for o in out]


class Gipformer:
    def __init__(self, model="g-group-ai-lab/gipformer-68M-rnnt", revision=None, int8=False, num_threads=2, **_):
        import sherpa_onnx
        from huggingface_hub import snapshot_download

        d = Path(snapshot_download(model, revision=revision, allow_patterns=["*.onnx", "tokens.txt", "config.json"]))
        from .common import sha256_file
        sfx = ".int8.onnx" if int8 else ".onnx"
        for name in ["encoder" + sfx, "decoder" + sfx, "joiner" + sfx, "tokens.txt"]:
            if not (d / name).is_file(): raise FileNotFoundError(d / name)
        self.assets = {name: sha256_file(d / name) for name in ["encoder" + sfx, "decoder" + sfx, "joiner" + sfx, "tokens.txt"]}
        self.rec = sherpa_onnx.OfflineRecognizer.from_transducer(
            encoder=str(d / f"encoder{sfx}"), decoder=str(d / f"decoder{sfx}"), joiner=str(d / f"joiner{sfx}"),
            tokens=str(d / "tokens.txt"), num_threads=num_threads, sample_rate=16000, feature_dim=80, decoding_method="greedy_search")

    def transcribe(self, waves: list) -> list[str]:
        streams = []
        for w in waves:
            s = self.rec.create_stream()
            s.accept_waveform(16000, w)
            streams.append(s)
        self.rec.decode_streams(streams)
        return [s.result.text.strip() for s in streams]


def make_backend(cfg: dict):
    name = cfg.get("backend", "phowhisper")
    if name == "phowhisper":
        return PhoWhisper(**cfg)
    if name == "gipformer":
        return Gipformer(**cfg)
    raise ValueError(f"unknown ASR backend {name}")


def read_wave(path: Path):
    import numpy as np
    import soundfile as sf

    w, sr = sf.read(str(path), dtype="float32")
    if w.ndim > 1:
        w = w.mean(axis=1)
    if sr != 16000:
        import librosa

        w = librosa.resample(w, orig_sr=sr, target_sr=16000)
    return np.ascontiguousarray(w)


# ------------------------------------------------------------------ metrics
def norm_text(s: str) -> str:
    t = unicodedata.normalize("NFC", s or "").lower()
    t = re.sub(r"[^\w\s]", " ", t)
    return " ".join(t.split())


def edit_distance(a: list, b: list) -> int:
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y)))
        prev = cur
    return prev[-1]


_NUM_WORDS = {"không", "một", "mốt", "hai", "ba", "bốn", "tư", "năm", "lăm", "sáu", "bảy", "tám", "chín", "mười", "mươi",
              "trăm", "nghìn", "ngàn", "triệu", "tỷ", "tỉ", "rưỡi", "linh", "lẻ"}
_TIME_WORDS = {"giờ", "phút", "sáng", "trưa", "chiều", "tối", "đêm", "mai", "nay", "mốt", "tuần", "tháng", "thứ", "chủ", "nhật"}


def entity_tokens(gold: str) -> dict[str, list[str]]:
    """Gold tokens that are numbers, time words, or proper names (capitalised, not sentence-initial)."""
    raw = re.findall(r"\w+", unicodedata.normalize("NFC", gold or ""))
    names = [w.lower() for i, w in enumerate(raw) if i > 0 and w[:1].isupper()]
    toks = norm_text(gold).split()
    return {
        "number": [t for t in toks if t.isdigit() or t in _NUM_WORDS],
        "time": [t for t in toks if t in _TIME_WORDS],
        "name": names,
    }


def score_pairs(pairs: list[tuple[str, str]]) -> dict:
    """pairs of (gold, hypothesis): corpus WER/CER and per-entity-class token recall."""
    we = wn = ce = cn = 0
    ent = {k: [0, 0] for k in ("number", "time", "name")}
    for g, h in pairs:
        gt, ht = norm_text(g).split(), norm_text(h).split()
        we += edit_distance(gt, ht)
        wn += len(gt)
        gc, hc = list(norm_text(g).replace(" ", "")), list(norm_text(h).replace(" ", ""))
        ce += edit_distance(gc, hc)
        cn += len(gc)
        hset = set(ht)
        for k, toks in entity_tokens(g).items():
            ent[k][0] += sum(t in hset for t in toks)
            ent[k][1] += len(toks)
    return {
        "n": len(pairs),
        "wer": we / wn if wn else None,
        "word_edits": we, "gold_words": wn,
        "cer": ce / cn if cn else None,
        "char_edits": ce, "gold_chars": cn,
        "entity_recall": {k: (a / b if b else None) for k, (a, b) in ent.items()},
        "entity_counts": {k: b for k, (a, b) in ent.items()},
    }


def gold_texts(rows: list[dict]) -> dict[str, str]:
    out = {}
    for r in rows:
        for t in r.get("history", []):
            if t.get("role") == "user" and t.get("audio") and t.get("audio_text") is not None:
                if t["audio"] in out and out[t["audio"]] != t["audio_text"]: raise ValueError("conflicting gold transcript")
                out[t["audio"]] = t["audio_text"]
        if r.get("audio_text") is not None:
            if r["audio"] in out and out[r["audio"]] != r["audio_text"]: raise ValueError("conflicting gold transcript")
            out[r["audio"]] = r["audio_text"]
    return out


# ------------------------------------------------------------------ worker
def run_hash(cfg: dict) -> str:
    return config_hash(cfg, ignore=RUN_IGNORE)


def cache_name(cfg: dict) -> str:
    return f"asr_{cfg['split']}_{cfg['asr']['backend']}"


def legacy_work(cfg: dict, shard_index: int) -> dict:
    set_thread_env(int(cfg.get("cpu_threads", 1)))
    deadline = Deadline(float(cfg.get("wallclock_s", 3.5 * 3600)))
    split_dir = resolve_split_dir(cfg)
    rows = load_rows(split_dir)
    if cfg.get("subset_ids"):
        keep = set(cfg["subset_ids"])
        rows = [r for r in rows if r["id"] in keep]
    if cfg.get("limit"):
        rows = rows[: int(cfg["limit"])]
    paths = sorted({p for r in rows for p in audio_paths(r)})
    n = int(cfg.get("num_shards", 1))
    mine = [p for p in paths if shard_of(p, n) == shard_index]
    out = Path(cfg["out_dir"])
    cache = DurableJsonl(out / f"{cache_name(cfg)}.shard{shard_index}.jsonl", run_hash(cfg))
    todo = [p for p in mine if p not in cache.done_ids()]
    t0 = time.time()
    backend = None
    gold = gold_texts(rows) if cfg["asr"]["backend"] == "gold" else None
    if gold is None:
        backend = make_backend(cfg["asr"])
    load_s = time.time() - t0
    bs = int(cfg["asr"].get("batch_size", 16))
    audio_s = 0.0
    t_run = time.time()
    stopped = False
    for b in range(0, len(todo), bs):
        if deadline.expired():
            stopped = True
            break
        chunk = todo[b : b + bs]
        try:
            if gold is not None:
                texts, secs = [gold.get(p, "") for p in chunk], [0.0] * len(chunk)
            else:
                waves = [read_wave(split_dir / p) for p in chunk]
                secs = [len(w) / 16000 for w in waves]
                tb = time.time()
                texts = backend.transcribe(waves)
                per = (time.time() - tb) / len(chunk)
            for p, t, s in zip(chunk, texts, secs):
                cache.append({"id": p, "status": "completed", "text": t, "audio_s": round(s, 2), "asr_s": round(per, 3) if gold is None else 0})
                audio_s += s
        except Exception as e:
            for p in chunk:
                cache.append({"id": p, "status": "retryable", "error": f"{type(e).__name__}: {e}"})
        print(f"[asr shard {shard_index}] {b + len(chunk)}/{len(todo)}", flush=True)
    cache.close()
    peak = None
    try:
        import torch

        peak = round(torch.cuda.max_memory_allocated() / 2**30, 2) if torch.cuda.is_available() else None
    except Exception:
        pass
    summary = {"shard": shard_index, "audios_in_shard": len(mine), "todo": len(todo), "stopped_early_by_wallclock": stopped,
               "model_load_s": round(load_s, 1), "run_s": round(time.time() - t_run, 1), "audio_s": round(audio_s, 1),
               "rtf": round((time.time() - t_run) / max(audio_s, 1e-9), 4), "peak_vram_gb": peak, "env": env_info()}
    atomic_write_json(out / f"{cache_name(cfg)}.summary.shard{shard_index}.json", summary)
    return summary


def load_cache(dirs, split: str | None = None, backend: str | None = None) -> dict[str, str]:
    """{audio relative path: text} from every asr_*.shard*.jsonl under the given dirs (output of this notebook, or
    an attached output version)."""
    texts = {}
    pat = f"asr_{split or '*'}_{backend or '*'}.shard*.jsonl"
    for d in dirs:
        for p in sorted(Path(d).rglob(pat)):
            rows, _ = read_jsonl_tolerant(p)
            for r in rows[1:]:
                if r.get("status") == "completed":
                    texts[r["id"]] = r["text"]
    return texts


def evaluate(cfg: dict) -> dict:
    split_dir = resolve_split_dir(cfg)
    rows = load_rows(split_dir)
    gold = gold_texts(rows)
    hyp = load_cache([cfg["out_dir"]], cfg["split"], cfg["asr"]["backend"])
    pairs = [(gold[p], hyp[p]) for p in sorted(gold) if p in hyp]
    rep = score_pairs(pairs)
    worst = sorted(((edit_distance(norm_text(g).split(), norm_text(h).split()) / max(len(norm_text(g).split()), 1), p, g, h)
                    for p, (g, h) in zip([p for p in sorted(gold) if p in hyp], pairs)), reverse=True)[:30]
    rep["worst"] = [{"audio": p, "wer": round(w, 3), "gold": g, "hyp": h} for w, p, g, h in worst]
    atomic_write_json(Path(cfg["out_dir"]) / f"{cache_name(cfg)}.metrics.json", rep)
    return rep


def launch(cfg_path: str, cfg: dict, gpus: list[int], lib_root: str, module: str = "lmlib.asr", python: str | None = None) -> list[int]:
    out = Path(cfg["out_dir"])
    out.mkdir(parents=True, exist_ok=True)
    procs = []
    for i in range(int(cfg.get("num_shards", 1))):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpus[i % len(gpus)]) if gpus else "", PYTHONUNBUFFERED="1",
                   PYTHONPATH=lib_root + os.pathsep + os.environ.get("PYTHONPATH", ""))
        log = open(out / f"{module.split('.')[-1]}_worker{i}.log", "a", encoding="utf-8")
        procs.append((subprocess.Popen([python or sys.executable, "-m", module, "--config", cfg_path, "--shard", str(i)], env=env, stdout=log, stderr=subprocess.STDOUT), log))
    codes = []
    for p, log in procs:
        codes.append(p.wait())
        log.close()
    return codes



# Strict dev cache contract. Legacy public/SFT caches remain readable only in their legacy presets.
def asr_components(backend):
    from .provenance import source_hash
    from .data import prepare_audio_manifest
    cls = PhoWhisper if backend == 'phowhisper' else Gipformer
    return {'read_wave':source_hash(read_wave), 'backend':source_hash(cls),
            'audio_selection':source_hash(prepare_audio_manifest), 'worker':source_hash(work)}


def asr_cache_key(manifest, backend_cfg, component_hashes):
    from .provenance import digest, RUNTIME_KEYS
    return digest({'audio_manifest_sha256':manifest['audio_manifest_sha256'],
                   'backend_config':{k:v for k,v in backend_cfg.items() if k not in RUNTIME_KEYS},
                   'components':component_hashes, 'preprocess':'mono-float32-16khz-v1'})


def cache_expectation(manifest, backend_cfg, execution=None):
    components=asr_components(backend_cfg['backend'])
    return {'schema':'asr_cache_v1','content_key':asr_cache_key(manifest,backend_cfg,components),
            'audio_manifest_sha256':manifest['audio_manifest_sha256'], 'backend':backend_cfg['backend'],
            'model':backend_cfg['model'],'revision':backend_cfg['revision'],
            'decoding':backend_cfg,'preprocess':'mono-float32-16khz-v1','component_hashes':components,
            'source_execution':execution}


def load_verified_transcripts(cache_dir, expected, audio_ids, allow_partial=False):
    from .provenance import collect_shards
    root=Path(cache_dir);p=root/'cache_manifest.json'
    if not p.is_file():raise ValueError(f'no verified ASR cache manifest: {root}')
    got=json.loads(p.read_text())
    for k in ('schema','content_key','audio_manifest_sha256','backend'):
        if not expected.get(k) or got.get(k)!=expected[k]:raise ValueError(f'ASR cache mismatch: {k}')
    records=collect_shards(root,'transcripts.shard*.jsonl',allowed_ids=audio_ids)
    for f in root.glob('transcripts.shard*.jsonl'):
        rows,_=read_jsonl_tolerant(f);h=rows[0]['_header']
        if h.get('content_key',h.get('run_hash'))!=expected['content_key']:raise ValueError('ASR shard content mismatch')
    texts={i:r['text'] for i,r in records.items() if r.get('status')=='completed' and isinstance(r.get('text'),str)}
    missing=set(audio_ids)-set(texts)
    if missing and not allow_partial:raise ValueError(f'missing ASR audio, including history: {sorted(missing)[:8]}')
    return texts


def work(cfg, shard_index, backend=None):
    if not cfg.get('strict_dev'):return legacy_work(cfg,shard_index)
    from .common import session_deadline, can_start, MemoryMonitor
    from .provenance import digest, freeze_json
    from .data import prepare_audio_manifest
    set_thread_env(int(cfg.get('cpu_threads',1)))
    deadline=session_deadline(cfg);monitor=MemoryMonitor();out=Path(cfg['out_dir'])
    sd=resolve_split_dir(cfg);rows=load_rows(sd)
    manifest=prepare_audio_manifest(rows,cfg['subset_ids'],sd)
    expected=cache_expectation(manifest,cfg['asr'],cfg.get('execution_id'))
    # Source execution is provenance, not a semantic constraint when resuming.
    manifest_path=out/'cache_manifest.json'
    if manifest_path.exists():
        previous=json.loads(manifest_path.read_text())
        if previous['content_key']!=expected['content_key']:raise ValueError('ASR cache changed')
    else:atomic_write_json(manifest_path,expected)
    freeze_json(out/'audio_manifest.json',manifest)
    n=int(cfg['num_shards']);mine=[p for p in manifest['audio_ids'] if shard_of(p,n)==shard_index]
    rh=digest({'content_key':expected['content_key'],'num_shards':n})
    cache=DurableJsonl(out/f'transcripts.shard{shard_index}.jsonl',rh,expected)
    todo=[p for p in mine if p not in cache.done_ids()]
    start=time.time();loaded_s=0.;audio_s=0.;completed=0;preprocess_s=decode_s=0.;batch_est=float(cfg.get('batch_estimate_s',30))
    try:
        if todo and can_start(deadline,batch_est):
            t=time.time();backend=backend or make_backend(cfg['asr']);loaded_s=time.time()-t
        for b in range(0,len(todo),int(cfg['asr'].get('batch_size',16))):
            if backend is None or not can_start(deadline,batch_est):break
            chunk=todo[b:b+int(cfg['asr'].get('batch_size',16))];t=time.time()
            try:
                waves=[read_wave(sd/p) for p in chunk];preprocess_s+=time.time()-t;tg=time.time()
                texts=backend.transcribe(waves);elapsed=time.time()-tg;decode_s+=elapsed
                if len(texts)!=len(chunk) or any(not isinstance(t,str) for t in texts):raise ValueError('ASR result cardinality/type mismatch')
                for p,text,w in zip(chunk,texts,waves):
                    duration=len(w)/16000;audio_s+=duration;completed+=1
                    cache.append({'id':p,'status':'completed','text':text,'audio_s':duration,
                                  'batch_decode_s':elapsed,'batch_size':len(chunk),'execution_id':cfg.get('execution_id')})
                batch_est=max(batch_est,(time.time()-t)*1.25)
            except Exception as e:
                for p in chunk:cache.append({'id':p,'status':'retryable','error':f'{type(e).__name__}: {e}'})
        summary={'shard':shard_index,'run_hash':rh,'expected':len(mine),'completed':len(cache.done_ids()),
                 'pending':len(set(mine)-cache.done_ids()),'model_load_s':loaded_s,'preprocess_s':preprocess_s,
                 'decode_s':decode_s,'run_s':time.time()-start,'audio_s':audio_s,
                 's_per_sample':(time.time()-start)/completed if completed else None,
                 'rtf':(time.time()-start)/audio_s if audio_s else None,
                 'throughput_audio_s_per_s':audio_s/(time.time()-start) if completed else None,
                 'peak_vram_mb':monitor.close(),'assets':getattr(backend,'assets',None),'execution_id':cfg.get('execution_id')}
        atomic_write_json(out/f'summary.shard{shard_index}.json',summary)
        return summary
    finally:cache.close();monitor.close()


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--shard", type=int, default=0)
    a = ap.parse_args(argv)
    cfg = json.loads(Path(a.config).read_text(encoding="utf-8"))
    print(json.dumps(work(cfg, a.shard), ensure_ascii=False, indent=1))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/lib/lmlib/agent.py
"""LifeMate agent loop: transcript + context + history (with tool_results) + installed-app schemas -> the LLM emits
one JSON action per step -> schema check (env.validate) -> env.call -> result goes into the next step.

Stops on {"done": true}, after sys.ask_missing_info, on a repeated identical call, after `max_invalid` rejected
proposals, at `max_steps`, or when the per-sample time budget runs out. The budget counts the sample's own share of
generation time (llm_s) + ASR, not wall-clock: samples are stepped together, so one batched step takes minutes on a
T4 and a wall-clock budget cut every sample after its first call (lm-B-004). The A30 SLA is measured separately. Every env.call is final (one ToolEnv per
sample); invalid proposals are never sent to env.call. Samples are stepped in batches so vLLM can batch them.
"""
from __future__ import annotations

import json
import re
import time
from dataclasses import dataclass, field

SYSTEM = """Bạn là trợ lý ảo điều khiển các ứng dụng trên điện thoại của người dùng Việt Nam bằng cách gọi tool.
Mỗi lượt bạn chỉ trả về ĐÚNG MỘT đối tượng JSON, không giải thích:
- gọi tool:   {"name": "<app>.<tool>", "arguments": {...}}
- kết thúc:   {"done": true}

Quy tắc:
1. Chỉ gọi tool có trong DANH SÁCH TOOL bên dưới (thuộc app đã cài) hoặc sys.ask_missing_info. Đúng tên tham số, kiểu, enum.
2. Không bao giờ bịa ID (contact_id, device_id, event_id...). Muốn có ID thì gọi tool đọc (read) trước rồi lấy ID từ kết quả.
3. Chỉ gọi tool đọc khi thật sự cần cho yêu cầu cuối. Nếu lịch sử (tool_results) đã có thông tin trả lời được thì dùng lại, không gọi lại.
4. Thiếu thông tin bắt buộc không suy ra được, hoặc kết quả tìm kiếm mơ hồ (nhiều người/đồ vật khớp): gọi
   {"name": "sys.ask_missing_info", "arguments": {"tool_name": "<tool cần gọi>", "params": ["<tham số thiếu>"]}} rồi dừng.
5. Yêu cầu ngoài khả năng của các app đã cài, tán gẫu, người dùng tự huỷ ("thôi khỏi"), hoặc yêu cầu xấu: trả về {"done": true} ngay, không gọi tool nào.
6. Thời gian dạng ISO 8601 có múi giờ +07:00 (vd 2026-11-28T07:30:00+07:00), ngày dạng YYYY-MM-DD; tính "mai", "tối nay", "thứ sáu tuần sau"... theo current_time.
7. Tham số tìm kiếm (query, tên người...) dùng đúng cụm từ người dùng nói. Văn bản tự do (nội dung tin nhắn, tiêu đề, ghi chú) viết ngắn gọn đúng ý người dùng.
8. Làm đủ mọi việc người dùng yêu cầu ở lượt cuối (có thể nhiều tool liên tiếp), xong thì {"done": true}.
"""


def _short(v, n: int) -> str:
    s = json.dumps(v, ensure_ascii=False, separators=(",", ":"))
    return s if len(s) <= n else s[: n - 1] + "…"


def _param_spec(name: str, p: dict, required: set) -> str:
    t = p.get("type", "")
    if t == "array":
        t = f"array<{(p.get('items') or {}).get('type', '')}>"
    bits = [f"{name}{'*' if name in required else ''}:{t}"]
    if p.get("enum"):
        bits.append("enum=" + "|".join(map(str, p["enum"])))
    if p.get("format"):
        bits.append(f"format={p['format']}")
    if "minimum" in p or "maximum" in p:
        bits.append(f"range=[{p.get('minimum', '')},{p.get('maximum', '')}]")
    if p.get("x-ref"):
        bits.append(f"id từ {p['x-ref']}")
    if p.get("x-match") == "fuzzy":
        bits.append("văn bản tự do")
    if p.get("description"):
        bits.append(p["description"][:80])
    return " ".join(bits)


class Catalog:
    def __init__(self, catalog: dict):
        self.tools = {}
        self.app_desc = {}
        for app in catalog["apps"]:
            self.app_desc[app["id"]] = f"{app.get('name_vi', '')}: {app.get('description', '')}"
            for t in app["tools"]:
                self.tools[t["name"]] = t
        self._render: dict[str, str] = {}

    def fuzzy_params(self, name: str) -> set[str]:
        props = ((self.tools.get(name) or {}).get("parameters") or {}).get("properties", {})
        return {k for k, v in props.items() if v.get("x-match") == "fuzzy"}

    def render_tool(self, name: str) -> str:
        if name not in self._render:
            t = self.tools[name]
            ps = t.get("parameters") or {}
            req = set(ps.get("required", []))
            params = "; ".join(_param_spec(k, v, req) for k, v in (ps.get("properties") or {}).items()) or "không tham số"
            one = ps.get("x-one-of")
            extra = f" | bắt buộc một trong: {one}" if one else ""
            self._render[name] = f"- {name} [{t.get('kind')}] {t.get('description', '')} | {params}{extra}"
        return self._render[name]

    def render_apps(self, installed: dict) -> str:
        lines = []
        for app, a in installed.items():
            lines.append(f"## {app} — {self.app_desc.get(app, '')}")
            lines += [self.render_tool(n) for n in a.get("tools", []) if n in self.tools]
        lines.append("## sys")
        if "sys.ask_missing_info" in self.tools:
            lines.append(self.render_tool("sys.ask_missing_info"))
        return "\n".join(lines)


def render_history(row: dict, transcripts: dict[str, str], result_chars: int) -> str:
    out = []
    for t in row.get("history", []):
        if t.get("role") == "user":
            out.append(f"Người dùng: {transcripts.get(t.get('audio'), t.get('audio_text') or '(không nghe rõ)')}")
        else:
            out.append(f"Trợ lý: {t.get('text', '')}")
            for c, r in zip(t.get("tool_calls") or [], (t.get("tool_results") or []) + [None] * 10):
                out.append(f"  đã gọi {_short(c, 400)}")
                if r is not None:
                    out.append(f"  kết quả {_short(r, result_chars)}")
    return "\n".join(out) or "(không có)"


def build_messages(row: dict, request_text: str, catalog: Catalog, transcripts: dict, steps: list[dict], result_chars: int = 1500) -> list[dict]:
    ctx = row["context"]
    head = {k: ctx.get(k) for k in ("current_time", "timezone", "location", "user_profile")}
    parts = [
        "# NGỮ CẢNH\n" + json.dumps(head, ensure_ascii=False),
        "# DANH SÁCH TOOL (app đã cài)\n" + catalog.render_apps(ctx.get("installed_apps", {})),
        "# LỊCH SỬ HỘI THOẠI\n" + render_history(row, transcripts, result_chars),
        f"# YÊU CẦU CUỐI CỦA NGƯỜI DÙNG\n{request_text}",
    ]
    if steps:
        lines = []
        for i, s in enumerate(steps, 1):
            if s["kind"] == "call":
                lines.append(f"{i}. đã gọi {_short({'name': s['name'], 'arguments': s['arguments']}, 600)}\n   kết quả {_short(s['result'], result_chars)}")
            else:
                lines.append(f"{i}. đề xuất bị từ chối (không gọi): {_short(s.get('proposal'), 300)} — lỗi: {s['error']}")
        parts.append("# CÁC BƯỚC ĐÃ LÀM CHO YÊU CẦU CUỐI\n" + "\n".join(lines))
    parts.append("Trả về JSON cho bước tiếp theo:")
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "\n\n".join(parts)}]


def _first_json_object(text: str):
    text = re.sub(r"<think>.*?</think>", "", text or "", flags=re.S)
    m = re.search(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", text, re.S)
    if m:
        text = m.group(1)
    start = text.find("{")
    while start != -1:
        depth, in_str, esc = 0, False, False
        for i in range(start, len(text)):
            ch = text[i]
            if in_str:
                esc = (ch == "\\") and not esc
                if ch == '"' and not esc:
                    in_str = False
                continue
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    try:
                        return json.loads(text[start : i + 1])
                    except json.JSONDecodeError:
                        break
        start = text.find("{", start + 1)
    return None


def parse_action(text: str):
    """('call', name, args) | ('done', None, None) | ('invalid', None, reason)."""
    obj = _first_json_object(text)
    if not isinstance(obj, dict):
        return ("invalid", None, "không đọc được JSON")
    if obj.get("done") is True or obj.get("name") in ("done", "finish"):
        return ("done", None, None)
    name = obj.get("name") or obj.get("tool")
    args = obj.get("arguments", obj.get("args", obj.get("parameters", {})))
    if isinstance(args, str):
        try:
            args = json.loads(args)
        except json.JSONDecodeError:
            return ("invalid", None, "arguments không phải JSON object")
    if not isinstance(name, str) or not isinstance(args, dict):
        return ("invalid", None, "thiếu name hoặc arguments")
    return ("call", name, args)


@dataclass
class AgentConfig:
    max_steps: int = 8
    max_new_tokens: int = 256
    max_invalid: int = 2
    sample_budget_s: float = 30.0
    result_chars: int = 1500
    budget_mode: str = "amortized"  # legacy public proxy; strict paired-dev uses admitted wall time


@dataclass
class Episode:
    row: dict
    request: str
    env: object
    t0: float
    asr_s: float = 0.0
    steps: list = field(default_factory=list)
    invalid: int = 0
    finished: str | None = None  # done | ask | repeat | max_steps | invalid | budget | too_long
    gen_tokens: int = 0
    llm_s: float = 0.0


def run_episodes(episodes: list[Episode], llm, catalog: Catalog, transcripts: dict, cfg: AgentConfig) -> None:
    for step in range(cfg.max_steps + cfg.max_invalid + 1):
        active = [e for e in episodes if e.finished is None]
        if not active:
            return
        for e in active:
            n_calls = sum(s["kind"] == "call" for s in e.steps)
            if n_calls >= cfg.max_steps:
                e.finished = "max_steps"
            elif (time.time() - e.t0 if cfg.budget_mode == "wall" else e.llm_s + e.asr_s) > cfg.sample_budget_s:
                e.finished = "budget"
        active = [e for e in active if e.finished is None]
        if not active:
            return
        convs = [build_messages(e.row, e.request, catalog, transcripts, e.steps, cfg.result_chars) for e in active]
        t = time.time()
        outs = llm.generate(convs, cfg.max_new_tokens)
        if len(outs) != len(active) or any(o.finish == "error" for o in outs):
            raise RuntimeError("LLM backend failed or returned the wrong number of outputs")
        dt = (time.time() - t) / len(active)
        for e, o in zip(active, outs):
            e.llm_s += dt
            e.gen_tokens += o.output_tokens
            if o.finish == "too_long":
                e.finished = "too_long"
                continue
            kind, name, args = parse_action(o.text)
            if kind == "done":
                e.finished = "done"
                continue
            if kind == "invalid":
                e.steps.append({"kind": "invalid", "proposal": o.text[:300], "error": args})
                e.invalid += 1
                if e.invalid >= cfg.max_invalid:
                    e.finished = "invalid"
                continue
            if any(s["kind"] == "call" and s["name"] == name and s["arguments"] == args for s in e.steps):
                e.finished = "repeat"
                continue
            chk = e.env.validate(name, args)
            if not chk.get("ok"):
                e.steps.append({"kind": "invalid", "proposal": {"name": name, "arguments": args}, "error": chk.get("error")})
                e.invalid += 1
                if e.invalid >= cfg.max_invalid:
                    e.finished = "invalid"
                continue
            result = e.env.call(name, args)
            e.steps.append({"kind": "call", "name": name, "arguments": args, "result": result})
            if name == "sys.ask_missing_info":
                e.finished = "ask"
    for e in episodes:
        if e.finished is None:
            e.finished = "max_steps"

In [ ]:
%%writefile /kaggle/working/lib/lmlib/scorer.py
"""Local LifeMate scorer for train/dev: hash gate + call F1 (rulebook formula).

Gold hash = replay label.tool_calls on a scorer-only session (separate from inference). Call matching: same name,
same argument keys; fuzzy (x-match) params match when token-F1 > 0.6 after NFC+lower; other values compare like the
simulator's _eq (strings lower-cased, spaces collapsed). Assumption: optional params present on one side only make the
call unmatched (not confirmed against the organisers' matcher).
"""
from __future__ import annotations

import re
import unicodedata
from collections import Counter


def _tok(s) -> list[str]:
    return re.findall(r"\w+", unicodedata.normalize("NFC", str(s)).lower())


def token_f1(a, b) -> float:
    ta, tb = _tok(a), _tok(b)
    if not ta and not tb:
        return 1.0
    common = sum((Counter(ta) & Counter(tb)).values())
    if not common:
        return 0.0
    p, r = common / len(ta), common / len(tb)
    return 2 * p * r / (p + r)


def _val_eq(a, b) -> bool:
    if isinstance(a, str) and isinstance(b, str):
        return " ".join(a.lower().split()) == " ".join(b.lower().split())
    if isinstance(a, list) and isinstance(b, list):
        return len(a) == len(b) and all(_val_eq(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_val_eq(a[k], b[k]) for k in a)
    if isinstance(a, bool) or isinstance(b, bool):
        return a is b
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        return abs(a - b) < 1e-9
    return a == b


def call_match(pred: dict, gold: dict, fuzzy: set) -> bool:
    if pred.get("name") != gold.get("name"):
        return False
    pa, ga = pred.get("arguments") or {}, gold.get("arguments") or {}
    if set(pa) != set(ga):
        return False
    for k, gv in ga.items():
        if k in fuzzy:
            if token_f1(pa[k], gv) <= 0.6:
                return False
        elif not _val_eq(pa[k], gv):
            return False
    return True


def count_matches(pred_calls: list, gold_calls: list, fuzzy_of) -> int:
    used = [False] * len(gold_calls)
    m = 0
    for p in pred_calls:
        for j, g in enumerate(gold_calls):
            if not used[j] and call_match(p, g, fuzzy_of(g.get("name"))):
                used[j] = True
                m += 1
                break
    return m


def sample_score(pred_calls, pred_hash, gold_calls, gold_hash, fuzzy_of) -> dict:
    hash_ok = pred_hash == gold_hash
    if not gold_calls:
        s = 1.0 if (hash_ok and not pred_calls) else 0.0
        return {"score": s, "hash_ok": hash_ok, "M": 0, "U": len(pred_calls), "G": 0, "tool_f1_before_gate": 1.0 if not pred_calls else 0.0}
    m = count_matches(pred_calls, gold_calls, fuzzy_of)
    u = len(pred_calls) - m
    if not hash_ok or m == 0:
        s = 0.0
    else:
        p, r = m / (m + u), m / len(gold_calls)
        s = 2 * p * r / (p + r)
    return {"score": s, "hash_ok": hash_ok, "M": m, "U": u, "G": len(gold_calls), "tool_f1_before_gate": 2*m/(len(pred_calls)+len(gold_calls))}


def gold_hash(ToolEnv, split: str, sample_id: str, root, gold_calls: list) -> str:
    """Replay the label on a fresh scorer-only session."""
    s = ToolEnv(split, sample_id, root=root)
    for c in gold_calls:
        s.call(c["name"], c.get("arguments") or {})
    return s.get_state_hash()

In [ ]:
%%writefile /kaggle/working/lib/lmlib/run.py
"""Lane B worker: `python -m lmlib.run --config cfg.json --shard I`.

mode diagnostic: train ids, clean audio_text transcripts (separates planner errors from ASR errors; not end-to-end)
mode e2e:        train ids, transcripts from the ASR cache (cfg.asr_cache_dirs)
mode public:     every id of the split (public_test), transcripts from the ASR cache; output -> submission.jsonl
One durable record per sample (tool_calls + state_hash straight from the sample's ToolEnv); a sample interrupted
mid-way is redone from a fresh ToolEnv in the next session. Samples whose ToolEnv cannot be created stay retryable.
"""
from __future__ import annotations

import argparse
import json
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path

from .agent import AgentConfig, Catalog, Episode, run_episodes
from .asr import gold_texts, launch as _launch, load_cache
from .common import DurableJsonl, Deadline, atomic_write_json, atomic_write_text, config_hash, dumps, env_info, read_jsonl_tolerant, set_thread_env, shard_of
from .data import make_manifest, load_rows, resolve_split_dir, audio_paths
from .provenance import semantic_hash, collect_shards
from .common import session_deadline, can_start, MemoryMonitor

RUN_IGNORE = ("shard_index", "num_shards", "out_dir", "resume", "wallclock_s", "parent_run", "notes", "input_roots", "split_dir", "asr_cache_dirs")


def import_zaic(split_dir: Path):
    """zaic_tools of this split (LIVE_APPS / FREE_TEXT_KEYS differ between splits)."""
    sys.path.insert(0, str(split_dir))
    for m in [m for m in sys.modules if m == "zaic_tools" or m.startswith("zaic_tools.")]:
        del sys.modules[m]
    import zaic_tools

    return zaic_tools


def select_rows(cfg: dict, rows: list[dict]) -> list[dict]:
    subset = cfg.get("subset", "all")
    if subset != "all":
        if isinstance(subset, list):
            keep = set(subset)
        else:
            ids = make_manifest(rows, seed=cfg.get("manifest_seed", 2026))["ids"]
            keep = set().union(*(ids[s] for s in subset.split("+")))
        rows = [r for r in rows if r["id"] in keep]
    if cfg.get("limit"):
        rows = rows[: int(cfg["limit"])]
    return rows


def transcripts_for(cfg: dict, rows: list[dict]) -> dict[str, str]:
    if cfg["mode"] == "diagnostic":
        return gold_texts(rows)
    if cfg.get("strict_dev"):
        from .asr import load_verified_transcripts
        need = sorted({p for row in rows for p in audio_paths(row)})
        # Verify the full cache once against its complete manifest, then select this shard's audio.
        expected = cfg['asr_expectation']
        import json
        root = Path(cfg['asr_cache_dirs'][0])
        manifest = json.loads((root/'audio_manifest.json').read_text())
        texts = load_verified_transcripts(root, expected, manifest['audio_ids'])
        if set(need) - set(texts): raise ValueError('missing history/final transcript')
        return {p:texts[p] for p in need}
    return load_cache(cfg.get("asr_cache_dirs", [cfg["out_dir"]]), cfg["split"], cfg.get("asr_backend"))



def inference_row(row: dict, mode: str) -> dict:
    import copy
    result=copy.deepcopy(row)
    result.pop('label',None)
    if mode != 'diagnostic':
        result.pop('audio_text',None)
        for turn in result.get('history',[]):turn.pop('audio_text',None)
    return result

def run_hash(cfg: dict) -> str:
    return semantic_hash(cfg)


def work(cfg: dict, shard_index: int, llm=None) -> dict:
    set_thread_env(int(cfg.get("cpu_threads", 1)))
    deadline = session_deadline(cfg)
    monitor = MemoryMonitor()
    split_dir = resolve_split_dir(cfg)
    zt = import_zaic(split_dir)
    root = split_dir.parent
    rows = select_rows(cfg, load_rows(split_dir))
    n = int(cfg.get("num_shards", 1))
    rows = [r for r in rows if shard_of(r["id"], n) == shard_index]
    out = Path(cfg["out_dir"])
    preds = DurableJsonl(out / f"preds.shard{shard_index}.jsonl", run_hash(cfg))
    traces = DurableJsonl(out / f"traces.shard{shard_index}.jsonl", run_hash(cfg))
    done = preds.done_ids() if cfg.get("resume", True) else set()
    todo = [r for r in rows if r["id"] not in done]
    texts = transcripts_for(cfg, rows)
    missing = [r["id"] for r in todo if any(p not in texts for p in audio_paths(r))]
    if missing and not cfg.get("allow_missing_transcripts", False):
        raise RuntimeError(f"{len(missing)} samples have no transcript for the final audio (e.g. {missing[:5]}); run ASR first")
    catalog = Catalog(json.loads((split_dir / f"{split_dir.name}_tools_catalog.json").read_text(encoding="utf-8")))
    acfg = AgentConfig(**cfg.get("agent", {}))
    t_load = time.time()
    if llm is None:
        from .llm import make_llm

        llm = make_llm(cfg["llm"])
    load_s = time.time() - t_load
    bs = int(cfg.get("admission_wave_size",cfg.get("batch_size", 16)))
    n_done, stopped, t_run = 0, False, time.time()
    for b in range(0, len(todo), bs):
        if not can_start(deadline,cfg.get("batch_estimate_s",0)):
            stopped = True
            break
        eps = []
        for r in todo[b : b + bs]:
            try:
                env = zt.ToolEnv(split_dir.name, r["id"], root=root)
            except Exception as e:
                preds.append({"id": r["id"], "status": "retryable", "error": f"ToolEnv: {type(e).__name__}: {e}"})
                continue
            eps.append(Episode(inference_row(r,cfg["mode"]), texts.get(r["audio"], ""), env, time.time(), asr_s=0.0 if cfg.get("strict_dev") else float(cfg.get("asr_s_per_sample",0))))
        run_episodes(eps, llm, catalog, texts, acfg)
        for e in eps:
            calls, h = e.env.get_tool_calls(), e.env.get_state_hash()
            preds.append({"id": e.row["id"], "status": "completed", "tool_calls": calls, "state_hash": h, "execution_id":cfg.get("execution_id")})
            traces.append({"id": e.row["id"], "status": "completed", "request": e.request, "finished": e.finished, "steps": e.steps,
                           "gen_tokens": e.gen_tokens, "llm_s": round(e.llm_s, 2), "wall_s": round(time.time() - e.t0, 2), "execution_id":cfg.get("execution_id"), "proposal_schema_errors":e.invalid})
            n_done += 1
        print(f"[agent shard {shard_index}] {b + len(eps)}/{len(todo)} elapsed {deadline.elapsed():.0f}s", flush=True)
    preds.close()
    traces.close()
    summary = {"shard": shard_index, "samples_in_shard": len(rows), "already_done": len(done), "completed_now": n_done,
               "stopped_early_by_wallclock": stopped, "missing_transcripts": len(missing), "model_load_s": round(load_s, 1),
               "run_hash":run_hash(cfg), "peak_vram_mb":monitor.close(), "execution_id":cfg.get("execution_id"),
               "run_s": round(time.time() - t_run, 1), "s_per_sample": round((time.time() - t_run) / max(n_done, 1), 2), "env": env_info()}
    atomic_write_json(out / f"summary.shard{shard_index}.json", summary)
    return summary


def launch(cfg_path: str, cfg: dict, gpus: list[int], lib_root: str, python: str | None = None) -> list[int]:
    """`python`: interpreter of the agent workers (the vLLM venv when it works)."""
    return _launch(cfg_path, cfg, gpus, lib_root, module="lmlib.run", python=python)


def merge(cfg: dict, finalize: bool = False) -> dict:
    """Validate before completing, score against the entire frozen manifest, never invent hashes."""
    split_dir=resolve_split_dir(cfg);zt=import_zaic(split_dir);out=Path(cfg['out_dir'])
    rows=select_rows(cfg,load_rows(split_dir));ids=[r['id'] for r in rows]
    raw=collect_shards(out,'preds.shard*.jsonl',run_hash(cfg),ids)
    traces=collect_shards(out,'traces.shard*.jsonl',run_hash(cfg),ids)
    catalog_json=json.loads((split_dir/f'{split_dir.name}_tools_catalog.json').read_text())
    catalog=Catalog(catalog_json);recs={};invalid={}
    for qid,rec in raw.items():
        if rec.get('status')!='completed':continue
        if not isinstance(rec.get('tool_calls'),list) or not isinstance(rec.get('state_hash'),str) or len(rec['state_hash'])!=64:
            invalid[qid]='invalid exported calls/hash';continue
        env=zt.ToolEnv(split_dir.name,qid,root=split_dir.parent)
        try:errors=[v for v in zt.schema_validate(rec['tool_calls'],catalog_json,env.allowed) if not v['ok']]
        except (KeyError,TypeError,ValueError) as e:errors=[str(e)]
        if errors:invalid[qid]=errors
        else:recs[qid]=rec
    pending=[i for i in ids if i not in recs]
    report={'n':len(ids),'expected_n':len(ids),'completed':len(recs),'completed_n':len(recs),
            'pending':len(pending),'pending_n':len(pending),'pending_head':pending[:20],
            'baseline_complete':not pending and not invalid,'run_hash':run_hash(cfg),
            'schema_errors':len(invalid),'output_schema_errors':invalid,
            'proposal_schema_errors':sum(t.get('proposal_schema_errors',0) for t in traces.values()),'submission':None}
    if report['baseline_complete']:
        text=''.join(dumps({k:recs[i][k] for k in ('id','tool_calls','state_hash')})+'\n' for i in ids)
        atomic_write_text(out/'submission.jsonl',text);report['submission']=str(out/'submission.jsonl')
        from .common import sha256_file
        report['prediction_sha256']=sha256_file(out/'submission.jsonl')
    elif (out/'submission.jsonl').exists():
        (out/'submission.jsonl').unlink()  # invalidate a stale final export; shard checkpoints remain intact
    if rows and 'label' in rows[0]:
        from .scorer import gold_hash,sample_score
        per=[];groups=defaultdict(list)
        for row in rows:
            qid=row['id'];rec=recs.get(qid);gold=row['label']['tool_calls']
            if rec:
                sc=sample_score(rec['tool_calls'],rec['state_hash'],gold,gold_hash(zt.ToolEnv,split_dir.name,qid,split_dir.parent,gold),catalog.fuzzy_params)
            else:sc={'score':0.,'hash_ok':False,'tool_f1_before_gate':0.,'M':0,'U':0,'G':len(gold)}
            kinds={catalog.tools.get(c['name'],{}).get('kind') for c in gold}
            group='empty' if not gold else ('ask' if any(c['name']=='sys.ask_missing_info' for c in gold) else ('write' if 'write' in kinds else 'read_only'))
            sc.update(id=qid,pending=rec is None,group=group,has_history=bool(row.get('history')),finished=traces.get(qid,{}).get('finished'))
            per.append(sc);groups[group].append(sc['score']);groups['history' if sc['has_history'] else 'no_history'].append(sc['score'])
        n=len(ids);total=sum(s['score'] for s in per)
        report.update(score=100*total/n if n else None,score_full_manifest=100*total/n if n else None,
                      score_completed_only=100*total/len(recs) if recs else None,
                      hash_ok_rate=sum(s['hash_ok'] for s in per)/n if n else None,
                      hash_ok_rate_full_manifest=sum(s['hash_ok'] for s in per)/n if n else None,
                      tool_f1_before_gate=sum(s['tool_f1_before_gate'] for s in per)/n if n else None,
                      by_group={k:{'n':len(v),'score':100*sum(v)/len(v) if v else None} for k,v in groups.items()},
                      finished_counts=dict(Counter(s['finished'] for s in per)))
        atomic_write_text(out/'per_id_scores.jsonl',''.join(dumps(s)+'\n' for s in per))
    atomic_write_json(out/'merge_report.json',report)
    return report


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--shard", type=int, default=0)
    a = ap.parse_args(argv)
    cfg = json.loads(Path(a.config).read_text(encoding="utf-8"))
    print(json.dumps(work(cfg, a.shard), ensure_ascii=False, indent=1))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/lib/lmlib/sft.py
"""Round-2 SFT/LoRA on real train trajectories, with exact resume across Kaggle sessions.

Examples: for every train_fit sample and every step k of label.tool_calls, the agent prompt (same builder as
inference, real tool_results of earlier steps) -> target JSON of call k; then -> {"done": true} unless the last call
is sys.ask_missing_info. Loss on the target tokens only.

Checkpoint (every `ckpt_every_steps` optimizer steps or `ckpt_every_min` minutes): LoRA adapter, optimizer, scheduler,
grad scaler, RNG states, epoch / cursor in the fixed per-epoch permutation, global step, loss log, config hash.
Written to ckpt-XXXXXX.partial, checksummed, COMPLETE marker, then renamed; the latest two are kept. Base weights are
never copied into checkpoints.
"""
from __future__ import annotations

import argparse
import glob
import json
import math
import os
import random
import shutil
import time
from pathlib import Path

from .agent import Catalog, build_messages
from .asr import gold_texts
from .common import atomic_write_json, config_hash, sha256_file


def build_examples(rows: list[dict], catalog: Catalog, result_chars: int = 1500, texts: dict | None = None) -> list[dict]:
    """texts: {audio path: transcript} for the prompts (an ASR cache); None or a missing audio -> gold audio_text."""
    texts = {**gold_texts(rows), **(texts or {})}
    out = []
    for r in rows:
        request = texts.get(r["audio"], r["audio_text"])
        calls, results = r["label"]["tool_calls"], r["label"]["tool_results"]
        steps = []
        for k, c in enumerate(calls):
            out.append({"id": f"{r['id']}#{k}", "messages": build_messages(r, request, catalog, texts, list(steps), result_chars),
                        "target": json.dumps({"name": c["name"], "arguments": c.get("arguments") or {}}, ensure_ascii=False)})
            steps.append({"kind": "call", "name": c["name"], "arguments": c.get("arguments") or {}, "result": results[k] if k < len(results) else {}})
        if not calls or calls[-1]["name"] != "sys.ask_missing_info":
            out.append({"id": f"{r['id']}#done", "messages": build_messages(r, request, catalog, texts, list(steps), result_chars),
                        "target": '{"done": true}'})
    return out


def tokenize(ex: dict, tok, max_len: int, thinking: bool = False):
    try:
        prompt = tok.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=True, enable_thinking=thinking)
    except TypeError:
        prompt = tok.apply_chat_template(ex["messages"], tokenize=False, add_generation_prompt=True)
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    t = tok(ex["target"] + (tok.eos_token or ""), add_special_tokens=False)["input_ids"]
    if len(p) + len(t) > max_len:
        return None
    return {"input_ids": p + t, "labels": [-100] * len(p) + t}


def _rng_state():
    import numpy as np
    import torch

    st = {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state()}
    if torch.cuda.is_available():
        st["cuda"] = torch.cuda.get_rng_state_all()
    return st


def _set_rng_state(st):
    import numpy as np
    import torch

    random.setstate(st["python"])
    np.random.set_state(st["numpy"])
    torch.set_rng_state(st["torch"])
    if "cuda" in st and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(st["cuda"])


def latest_checkpoint(dirs) -> Path | None:
    found = []
    for d in dirs:
        for p in glob.glob(str(Path(d) / "**" / "ckpt-*"), recursive=True):
            p = Path(p)
            if p.is_dir() and not p.name.endswith(".partial") and (p / "COMPLETE").exists():
                found.append((int(p.name.split("-")[1]), p))
    return max(found)[1] if found else None


def save_checkpoint(out_dir: Path, model, opt, sched, scaler, state: dict, keep: int = 2) -> Path:
    import torch
    from peft import get_peft_model_state_dict

    final = out_dir / f"ckpt-{state['global_step']:06d}"
    tmp = final.with_name(final.name + ".partial")
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True)
    torch.save(get_peft_model_state_dict(model), tmp / "adapter_state.pt")
    model.peft_config["default"].save_pretrained(str(tmp))
    torch.save({"optimizer": opt.state_dict(), "scheduler": sched.state_dict(), "scaler": scaler.state_dict() if scaler else None,
                "rng": _rng_state(), **state}, tmp / "training_state.pt")
    sums = {p.name: sha256_file(p) for p in sorted(tmp.iterdir())}
    atomic_write_json(tmp / "checksums.json", sums)
    (tmp / "COMPLETE").write_text(state["config_hash"])
    shutil.rmtree(final, ignore_errors=True)
    tmp.rename(final)
    ckpts = sorted(p for p in out_dir.glob("ckpt-*") if p.is_dir() and not p.name.endswith(".partial"))
    for old in ckpts[:-keep]:
        shutil.rmtree(old, ignore_errors=True)
    return final


def train(cfg: dict, examples: list[dict] | None = None, model=None, tok=None, log=print) -> dict:
    """cfg keys: model, revision, load_in_4bit, max_len, lora_r, lora_alpha, lora_dropout, target_modules, lr,
    warmup_ratio, epochs, micro_batch (1), grad_accum, max_steps, seed, ckpt_every_steps, ckpt_every_min,
    wallclock_s, out_dir, resume_dirs, gradient_checkpointing."""
    import numpy as np
    import torch
    from peft import LoraConfig, get_peft_model, set_peft_model_state_dict

    out_dir = Path(cfg["out_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)
    seed = int(cfg.get("seed", 0))
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    chash = config_hash(cfg, ignore=("out_dir", "resume_dirs", "wallclock_s", "ckpt_every_min", "max_steps_this_session"))
    if model is None:
        from transformers import AutoModelForCausalLM, AutoTokenizer

        tok = AutoTokenizer.from_pretrained(cfg["model"], revision=cfg.get("revision"))
        kw = {"revision": cfg.get("revision"), "device_map": {"": 0}}
        if cfg.get("load_in_4bit"):
            from transformers import BitsAndBytesConfig

            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
        else:
            kw["torch_dtype"] = torch.float16
        model = AutoModelForCausalLM.from_pretrained(cfg["model"], **kw)
        if cfg.get("load_in_4bit"):
            from peft import prepare_model_for_kbit_training

            model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=cfg.get("gradient_checkpointing", True))
        elif cfg.get("gradient_checkpointing", True):
            model.gradient_checkpointing_enable()
            model.enable_input_require_grads()
    lcfg = LoraConfig(r=cfg.get("lora_r", 16), lora_alpha=cfg.get("lora_alpha", 32), lora_dropout=cfg.get("lora_dropout", 0.05),
                      target_modules=cfg.get("target_modules", ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]), task_type="CAUSAL_LM")
    model = get_peft_model(model, lcfg)
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()  # fp32 master weights for LoRA; fp16 autocast for the forward
    data = [x for x in (tokenize(e, tok, int(cfg.get("max_len", 8192))) for e in examples) if x is not None]
    log(f"[sft] {len(data)}/{len(examples)} examples fit max_len")
    accum = int(cfg.get("grad_accum", 16))
    steps_per_epoch = math.ceil(len(data) / accum)
    total = int(cfg.get("max_steps") or steps_per_epoch * int(cfg.get("epochs", 1)))
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=float(cfg.get("lr", 1e-4)), weight_decay=0.0)
    warm = max(1, int(total * float(cfg.get("warmup_ratio", 0.03))))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(total - warm, 1)))
    use_scaler = torch.cuda.is_available() and not cfg.get("load_in_4bit_bf16", False)
    scaler = torch.amp.GradScaler("cuda") if use_scaler else None
    state = {"global_step": 0, "epoch": 0, "cursor": 0, "losses": [], "config_hash": chash}
    ck = latest_checkpoint([out_dir] + list(cfg.get("resume_dirs", [])))
    if ck is not None:
        st = torch.load(ck / "training_state.pt", map_location="cpu", weights_only=False)
        if st["config_hash"] != chash:
            raise RuntimeError(f"checkpoint {ck} was made with another config ({st['config_hash']} != {chash})")
        set_peft_model_state_dict(model, torch.load(ck / "adapter_state.pt", map_location="cpu"))
        opt.load_state_dict(st["optimizer"]); sched.load_state_dict(st["scheduler"])
        if scaler and st.get("scaler"):
            scaler.load_state_dict(st["scaler"])
        _set_rng_state(st["rng"])
        state = {k: st[k] for k in ("global_step", "epoch", "cursor", "losses", "config_hash")}
        log(f"[sft] resumed from {ck} at step {state['global_step']} epoch {state['epoch']} cursor {state['cursor']}")
    model.train()
    dev = next(model.parameters()).device
    t0 = last_ck = time.time()
    wall = float(cfg.get("wallclock_s", 3.5 * 3600))
    session_limit = cfg.get("max_steps_this_session")
    session_steps = 0
    stop_reason = "finished"
    while state["global_step"] < total:
        perm = np.random.RandomState(seed + state["epoch"]).permutation(len(data)).tolist()
        while state["cursor"] < len(perm) and state["global_step"] < total:
            idx = perm[state["cursor"] : state["cursor"] + accum]
            tot_tok = sum(sum(l != -100 for l in data[i]["labels"]) for i in idx)
            loss_sum = 0.0
            for i in idx:
                ids = torch.tensor([data[i]["input_ids"]], device=dev)
                n_tgt = sum(l != -100 for l in data[i]["labels"])
                keep = n_tgt + 1  # logits only where a target token is predicted (full-vocab logits of 6k tokens do not fit a T4)
                with torch.autocast("cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                    logits = model(input_ids=ids, logits_to_keep=keep).logits[:, :-1].float()
                tgt = torch.tensor([data[i]["labels"][-n_tgt:]], device=dev)
                loss = torch.nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1), ignore_index=-100, reduction="sum") / max(tot_tok, 1)
                (scaler.scale(loss) if scaler else loss).backward()
                loss_sum += float(loss)
            if scaler:
                scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            if scaler:
                scaler.step(opt); scaler.update()
            else:
                opt.step()
            opt.zero_grad(set_to_none=True)
            sched.step()
            state["cursor"] += len(idx)
            state["global_step"] += 1
            session_steps += 1
            state["losses"].append(round(loss_sum, 5))
            if state["global_step"] % 10 == 0:
                log(f"[sft] step {state['global_step']}/{total} loss {loss_sum:.4f} lr {sched.get_last_lr()[0]:.2e} {time.time() - t0:.0f}s")
            due = state["global_step"] % int(cfg.get("ckpt_every_steps", 50)) == 0 or time.time() - last_ck > 60 * float(cfg.get("ckpt_every_min", 20))
            stop = time.time() - t0 > wall or (session_limit and session_steps >= int(session_limit))
            if due or stop:
                save_checkpoint(out_dir, model, opt, sched, scaler, state)
                last_ck = time.time()
            if stop:
                stop_reason = "wallclock" if time.time() - t0 > wall else "session_step_limit"
                break
        if stop_reason != "finished":
            break
        if state["cursor"] >= len(data):
            state["epoch"] += 1
            state["cursor"] = 0
    if stop_reason == "finished":
        save_checkpoint(out_dir, model, opt, sched, scaler, state)
    summary = {"stop_reason": stop_reason, "global_step": state["global_step"], "total_steps": total, "epoch": state["epoch"],
               "examples": len(data), "last_losses": state["losses"][-10:], "session_s": round(time.time() - t0, 1), "config_hash": chash}
    atomic_write_json(out_dir / "train_summary.json", summary)
    summary["model"] = model
    return summary


def resume_selftest(out_dir: str, steps: int = 40) -> dict:
    """Tiny random model on CPU/GPU: `steps` straight vs `steps/2` + resume + `steps/2`; losses must agree."""
    import torch
    from transformers import LlamaConfig, LlamaForCausalLM

    def fresh():
        torch.manual_seed(0)
        m = LlamaForCausalLM(LlamaConfig(vocab_size=128, hidden_size=32, intermediate_size=64, num_hidden_layers=2, num_attention_heads=4, num_key_value_heads=4))
        return m.to("cuda" if torch.cuda.is_available() else "cpu")

    rnd = random.Random(0)
    ex = [{"input_ids": [rnd.randrange(128) for _ in range(24)]} for _ in range(64)]

    class Tok:
        eos_token = ""

        def apply_chat_template(self, m, **k):
            return m[0]["content"]

        def __call__(self, s, add_special_tokens=False):
            return {"input_ids": [int(x) for x in s.split()] if s else []}

    tok = Tok()
    examples = [{"messages": [{"role": "user", "content": " ".join(map(str, e["input_ids"][:12]))}], "target": " ".join(map(str, e["input_ids"][12:]))} for e in ex]
    base = {"max_len": 64, "lora_r": 4, "lora_alpha": 8, "lora_dropout": 0.0, "target_modules": ["q_proj", "v_proj"], "lr": 1e-3,
            "grad_accum": 2, "max_steps": steps, "seed": 0, "ckpt_every_steps": 1000, "gradient_checkpointing": False}
    a = train({**base, "out_dir": str(Path(out_dir) / "straight")}, examples, model=fresh(), tok=tok, log=lambda *_: None)
    train({**base, "out_dir": str(Path(out_dir) / "resumed"), "max_steps_this_session": steps // 2}, examples, model=fresh(), tok=tok, log=lambda *_: None)
    b = train({**base, "out_dir": str(Path(out_dir) / "resumed")}, examples, model=fresh(), tok=tok, log=lambda *_: None)
    la = json.loads(json.dumps(a["last_losses"]))
    lb = json.loads(json.dumps(b["last_losses"]))
    diff = max(abs(x - y) for x, y in zip(la, lb))
    return {"straight_last": la, "resumed_last": lb, "max_abs_diff": diff, "ok": diff < 1e-3 and a["global_step"] == b["global_step"] == steps}


def merge_adapter(base_model: str, ckpt: Path, out: Path, revision: str | None = None) -> Path:
    """Base fp16 + LoRA -> merged weights the agent notebook can load as `model`."""
    import torch
    from peft import LoraConfig, get_peft_model, set_peft_model_state_dict
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(base_model, revision=revision)
    m = AutoModelForCausalLM.from_pretrained(base_model, revision=revision, torch_dtype=torch.float16)
    m = get_peft_model(m, LoraConfig.from_pretrained(str(ckpt)))
    set_peft_model_state_dict(m, torch.load(ckpt / "adapter_state.pt", map_location="cpu"))
    m = m.merge_and_unload()
    m.save_pretrained(str(out), safe_serialization=True)
    tok.save_pretrained(str(out))
    return out


def merge_latest(base_model: str, ckpt_dirs, out: Path, revision: str | None = None) -> Path:
    """Merge the newest complete checkpoint found under `ckpt_dirs` (dir or list of dirs) into `out`."""
    ck = latest_checkpoint([ckpt_dirs] if isinstance(ckpt_dirs, (str, Path)) else ckpt_dirs)
    if ck is None:
        raise FileNotFoundError(f"no complete ckpt-* under {ckpt_dirs}")
    print(f"[sft] merging {ck} -> {out}", flush=True)
    return merge_adapter(base_model, ck, Path(out), revision)


def main(argv=None):
    """python -m lmlib.sft --config cfg.json  (cfg = train() keys + examples_path)"""
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    a = ap.parse_args(argv)
    cfg = json.loads(Path(a.config).read_text(encoding="utf-8"))
    with open(cfg["examples_path"], encoding="utf-8") as f:
        examples = [json.loads(l) for l in f if l.strip()]
    summary = train(cfg, examples, log=lambda *x: print(*x, flush=True))
    summary.pop("model", None)
    print(json.dumps(summary, ensure_ascii=False, indent=1))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/lib/lmlib/regression.py
"""CPU tests for the paired-dev and ASR experiments; no model download."""
import copy
import json
import tempfile
import unittest
from pathlib import Path
from . import data, asr
from .common import DurableJsonl, atomic_write_json
from .run import inference_row, run_hash

class LifeMateRegression(unittest.TestCase):
    def test_e2e_removes_gold_including_history(self):
        row={'id':'x','audio':'a.wav','audio_text':'GOLD','label':{'tool_calls':[]},'history':[{'role':'user','audio':'h.wav','audio_text':'GOLD_H'},{'role':'assistant','tool_results':[{'ok':True}]}]}
        result=inference_row(row,'e2e')
        self.assertNotIn('GOLD',json.dumps(result))
        self.assertEqual(row['audio_text'],'GOLD')
        self.assertEqual(result['history'][1]['tool_results'],[{'ok':True}])

    def test_resume_identity_excludes_execution_but_keeps_shards(self):
        cfg={'experiment':'x','experiment_id':'x','num_shards':2,'llm':{'revision':'r'},'agent':{'max_steps':8},'execution_id':'one','out_dir':'/a','wallclock_s':10}
        other=dict(cfg,execution_id='two',parent_execution_id='one',out_dir='/b',wallclock_s=20)
        self.assertEqual(run_hash(cfg),run_hash(other))
        self.assertNotEqual(run_hash(cfg),run_hash(dict(cfg,num_shards=1)))
        self.assertNotEqual(run_hash(cfg),run_hash(dict(cfg,agent={'max_steps':7})))

    def test_audio_manifest_deduplicates_and_rejects_conflicting_gold(self):
        with tempfile.TemporaryDirectory() as td:
            p=Path(td);(p/'a.wav').write_bytes(b'a');(p/'h.wav').write_bytes(b'h')
            rows=[{'id':'x','audio':'a.wav','audio_text':'a','history':[{'role':'user','audio':'h.wav','audio_text':'h'},{'role':'user','audio':'a.wav','audio_text':'a'}]}]
            m=data.prepare_audio_manifest(rows,['x'],p)
            self.assertEqual(m['audio_ids'],['a.wav','h.wav'])
            self.assertEqual(len(m['episode_roles']['a.wav']),2)
            rows[0]['history'][1]['audio_text']='conflict'
            with self.assertRaises(ValueError):data.prepare_audio_manifest(rows,['x'],p)

    def test_cache_checks_history_and_content_provenance(self):
        with tempfile.TemporaryDirectory() as td:
            expected={'schema':'asr_cache_v1','content_key':'key','audio_manifest_sha256':'audio','backend':'phowhisper'}
            atomic_write_json(Path(td)/'cache_manifest.json',expected)
            w=DurableJsonl(Path(td)/'transcripts.shard0.jsonl','key')
            w.append({'id':'a','status':'completed','text':''});w.close()
            self.assertEqual(asr.load_verified_transcripts(td,expected,['a']),{'a':''})
            with self.assertRaises(ValueError):asr.load_verified_transcripts(td,expected,['a','history'])
            with self.assertRaises(ValueError):asr.load_verified_transcripts(td,dict(expected,content_key='other'),['a'])

    def test_corpus_wer_counts_insertions_and_empty_denominator(self):
        r=asr.score_pairs([('a b','a'),('c d e f','c d e f')])
        self.assertAlmostEqual(r['wer'],1/6)
        self.assertEqual(r['word_edits'],1)
        self.assertEqual(r['gold_words'],6)
        self.assertIsNone(asr.score_pairs([('','insert')])['wer'])

    def test_cache_key_ignores_execution_and_driver(self):
        m={'audio_manifest_sha256':'a'};cfg={'backend':'phowhisper','revision':'r','execution_id':'one'}
        key=asr.asr_cache_key(m,cfg,{'read_wave':'w','decode':'d'})
        self.assertEqual(key,asr.asr_cache_key(m,dict(cfg,execution_id='two',out_dir='/b'),{'read_wave':'w','decode':'d'}))
        self.assertNotEqual(key,asr.asr_cache_key(m,dict(cfg,revision='s'),{'read_wave':'w','decode':'d'}))

    def test_asr_comparison_requires_both_complete_and_same_audio(self):
        from .asr_compare import compare_asr
        with tempfile.TemporaryDirectory() as td:
            root=Path(td);a=root/'control';b=root/'candidate';a.mkdir();b.mkdir()
            manifest={'audio_ids':['a'],'audio_manifest_sha256':'audio','episode_roles':{'a':[{'role':'final','episode_id':'x'}]}}
            atomic_write_json(root/'manifest.json',manifest)
            for folder,name,text in [(a,'phowhisper','a b'),(b,'gipformer','a')]:
                atomic_write_json(folder/'cache_manifest.json',{'schema':'asr_cache_v1','content_key':name,'audio_manifest_sha256':'audio','backend':name,'preprocess':'same'})
                w=DurableJsonl(folder/'transcripts.shard0.jsonl',name);w.append({'id':'a','status':'completed','text':text});w.close()
            r=compare_asr(a,b,root/'manifest.json',gold={'a':'a b'})
            self.assertTrue(r['comparison_complete']);self.assertEqual(r['delta_wer'],.5)
            (b/'transcripts.shard0.jsonl').unlink()
            r=compare_asr(a,b,root/'manifest.json',gold={'a':'a b'})
            self.assertFalse(r['comparison_complete']);self.assertNotIn('delta_wer',r)

    def test_pair_comparison_checks_common_fingerprint_and_ids(self):
        from .compare import compare_paired
        with tempfile.TemporaryDirectory() as td:
            root=Path(td);atomic_write_json(root/'manifest.json',{'selected_ids':['a','b']})
            for name,scores in [('diagnostic',[1,0]),('e2e',[0,1])]:
                d=root/name;d.mkdir()
                atomic_write_json(d/'comparison_contract.json',{'schema':'lm-paired-v1','common':'same','treatment':name})
                atomic_write_json(d/'merge_report.json',{'baseline_complete':True})
                (d/'per_id_scores.jsonl').write_text(''.join(json.dumps({'id':i,'score':v,'pending':False})+'\n' for i,v in zip(['a','b'],scores)))
            r=compare_paired(root/'diagnostic',root/'e2e',root/'manifest.json')
            self.assertEqual(r['mean_delta'],0);self.assertEqual(r['improved'],1);self.assertEqual(r['worsened'],1)
            atomic_write_json(root/'e2e'/'comparison_contract.json',{'schema':'lm-paired-v1','common':'changed','treatment':'e2e'})
            with self.assertRaises(ValueError):compare_paired(root/'diagnostic',root/'e2e',root/'manifest.json')

    def test_preflight_chooses_only_common_verified_config(self):
        from .preflight import choose_serving
        records=[{'trials':[{'capacity':1,'ok':True,'workload_s':10},{'capacity':2,'ok':True,'workload_s':9.8}]},
                 {'trials':[{'capacity':1,'ok':True,'workload_s':10},{'capacity':2,'ok':False,'workload_s':None}]}]
        self.assertEqual(choose_serving(records),1)
        records[1]['trials'][0]['ok']=False
        with self.assertRaises(RuntimeError):choose_serving(records)

if __name__=='__main__':unittest.main()


In [ ]:
%%writefile /kaggle/working/lib/lmlib/provenance.py
"""Content fingerprints and strict shard collection; execution metadata is separate."""
from __future__ import annotations
import hashlib
import inspect
import json
import shutil
from pathlib import Path
from .common import atomic_write_json, read_jsonl_tolerant, sha256_file

RUNTIME_KEYS = {'execution_id','parent_execution_id','account_alias','kaggle_version','timestamp','out_dir','input_roots','split_dir','cache_dir','asr_cache_dirs','resume_from_dirs','resume','wallclock_s','session_wallclock_s','session_started_epoch','session_deadline_epoch','export_reserve_s','batch_estimate_s','preflight_dirs','control_dir','candidate_dir','lib_root','llm_python','executor_python','num_gpus','shard_index','notes','parent_run','source_execution','source_version','asr_expectation','code_sha256'}

def digest(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,ensure_ascii=False,allow_nan=False,default=str,separators=(',',':')).encode()).hexdigest()

def source_hash(obj):
    return hashlib.sha256(inspect.getsource(obj).encode()).hexdigest()

def package_hash(package_dir):
    return digest({p.name:sha256_file(p) for p in sorted(Path(package_dir).glob('*.py')) if p.name not in {'regression.py'}})

def tree_hash(directory):
    root=Path(directory).resolve()
    records={}
    for p in sorted(root.rglob('*')):
        if p.is_file():
            if not p.resolve().is_relative_to(root):raise ValueError(f'source symlink escapes root: {p}')
            records[str(p.relative_to(root))]=sha256_file(p)
    return digest(records)

def semantic_hash(cfg):
    semantic={k:v for k,v in cfg.items() if k not in RUNTIME_KEYS}
    if cfg.get('experiment_id') and cfg.get('experiment') != cfg['experiment_id']:
        raise ValueError('experiment and experiment_id must agree')
    # Interpreter and source identities are content-addressed in preflight/manifest, never by mount path.
    semantic['code_sha256']=cfg.get('code_sha256') or package_hash(Path(__file__).parent)
    return digest(semantic)

def freeze_json(path,value):
    p=Path(path)
    if p.exists() and json.loads(p.read_text()) != value:raise ValueError(f'frozen artifact differs: {p}')
    atomic_write_json(p,value)
    return value

def collect_shards(out_dir,pattern,expected_hash=None,allowed_ids=None):
    records={};expected=set(allowed_ids) if allowed_ids is not None else None
    for p in sorted(Path(out_dir).glob(pattern)):
        rows,_=read_jsonl_tolerant(p)
        header=rows[0].get('_header') if rows else None
        if not header or (expected_hash is not None and header.get('run_hash')!=expected_hash):
            raise ValueError(f'shard header mismatch: {p}')
        for rec in rows[1:]:
            qid=rec.get('id')
            if expected is not None and qid not in expected:raise ValueError(f'unknown ID {qid} in {p}')
            previous=records.get(qid)
            if previous and previous.get('status')=='completed':
                if rec.get('status')!='completed' or digest({k:v for k,v in previous.items() if k not in {'execution_id'}})!=digest({k:v for k,v in rec.items() if k not in {'execution_id'}}):
                    raise ValueError(f'conflicting completed ID: {qid}')
            records[qid]=rec
    return records

def restore_outputs(dirs,destination):
    """Copy explicitly attached checkpoint roots, never scan all inputs or overwrite another lineage."""
    dest=Path(destination);dest.mkdir(parents=True,exist_ok=True)
    for directory in dirs:
        src=Path(directory)
        if not src.is_dir():raise FileNotFoundError(src)
        for p in src.rglob('*'):
            if not p.is_file() or p.suffix not in {'.json','.jsonl'}:continue
            target=dest/p.relative_to(src)
            if target.exists():
                if target.read_bytes()!=p.read_bytes():raise ValueError(f'resume destination already differs: {target}')
            else:
                target.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(p,target)


In [ ]:
%%writefile /kaggle/working/lib/lmlib/asr_compare.py
"""Compare complete ASR artifacts on exactly the same audio manifest."""
from __future__ import annotations
import json
from pathlib import Path
from .asr import load_verified_transcripts,score_pairs
from .common import atomic_write_json,atomic_write_text,dumps


def compare_asr(control_dir,candidate_dir,manifest_path,gold=None):
    manifest=json.loads(Path(manifest_path).read_text());ids=manifest['audio_ids']
    result={'comparison_complete':False,'baseline_complete':False,'n':len(ids),'delta_direction':'gipformer minus PhoWhisper'}
    folders=[Path(control_dir),Path(candidate_dir)];headers=[];texts=[]
    for d in folders:
        p=d/'cache_manifest.json'
        if not p.exists():
            result['waiting_for']=str(d);atomic_write_json(folders[1].parent/'asr_comparison.json',result);return result
        h=json.loads(p.read_text())
        if h.get('schema')!='asr_cache_v1' or h.get('audio_manifest_sha256')!=manifest['audio_manifest_sha256']:
            raise ValueError('ASR input/manifest provenance differs')
        headers.append(h);texts.append(load_verified_transcripts(d,h,ids,allow_partial=True))
    if headers[0].get('backend')!='phowhisper' or headers[1].get('backend')!='gipformer':raise ValueError('wrong ASR treatment')
    if headers[0].get('preprocess')!=headers[1].get('preprocess'):raise ValueError('ASR preprocessing differs')
    for component in ('read_wave','audio_selection'):
        if headers[0].get('component_hashes',{}).get(component)!=headers[1].get('component_hashes',{}).get(component):
            raise ValueError('ASR common component differs')
    result['completed']={name:len(t) for name,t in zip(['phowhisper','gipformer'],texts)}
    complete=all(set(t)==set(ids) for t in texts)
    if complete:
        if gold is None or set(ids)-set(gold):raise ValueError('missing reference transcripts')
        metrics=[score_pairs([(gold[i],t[i]) for i in ids]) for t in texts]
        per=[{'id':i,'roles':manifest.get('episode_roles',{}).get(i,[]),'gold':gold[i],
              'phowhisper':texts[0][i],'gipformer':texts[1][i],
              'phowhisper_metrics':score_pairs([(gold[i],texts[0][i])]),'gipformer_metrics':score_pairs([(gold[i],texts[1][i])])} for i in ids]
        result.update(comparison_complete=True,baseline_complete=True,control=metrics[0],candidate=metrics[1],
                      delta_wer=metrics[1]['wer']-metrics[0]['wer'] if all(m['wer'] is not None for m in metrics) else None,
                      delta_cer=metrics[1]['cer']-metrics[0]['cer'] if all(m['cer'] is not None for m in metrics) else None)
        result['timing']={}
        for name,d in zip(['phowhisper','gipformer'],folders):
            p=d/'phase_timing.json'
            result['timing'][name]=json.loads(p.read_text()) if p.is_file() else None
        result['by_role']={}
        for role in ('history','final'):
            subset=[i for i in ids if any(r['role'].startswith(role) for r in manifest.get('episode_roles',{}).get(i,[]))]
            result['by_role'][role]={name:score_pairs([(gold[i],t[i]) for i in subset]) for name,t in zip(['phowhisper','gipformer'],texts)}
        atomic_write_text(folders[1].parent/'asr_per_audio.jsonl',''.join(dumps(r)+'\n' for r in per))
    atomic_write_json(folders[1].parent/'asr_comparison.json',result)
    return result


In [ ]:
%%writefile /kaggle/working/lib/lmlib/compare.py
"""Paired clean/ASR analysis from artifacts; never loads a model."""
from __future__ import annotations
import json
import random
from pathlib import Path
from .common import atomic_write_json,atomic_write_text,load_jsonl,dumps
from .provenance import digest,RUNTIME_KEYS


def paired_contract(cfg,manifest):
    common={k:v for k,v in cfg.items() if k not in RUNTIME_KEYS|{'mode','asr_expectation'}}
    return {'schema':'lm-paired-v1','common':digest({'cfg':common,'manifest':manifest['manifest_sha256'],'input':manifest['input_sha256'],'code':manifest['code_sha256']}),'treatment':cfg['mode']}


def checked_scores(directory,ids):
    p=Path(directory)/'per_id_scores.jsonl'
    rows=load_jsonl(p) if p.exists() else [];out={}
    for row in rows:
        if row['id'] not in ids:raise ValueError('unknown comparison ID')
        if row['id'] in out:raise ValueError('duplicate comparison ID')
        out[row['id']]=row
    return out


def compare_paired(clean_dir,e2e_dir,manifest_path):
    clean,e2e=Path(clean_dir),Path(e2e_dir);manifest=json.loads(Path(manifest_path).read_text());ids=manifest['selected_ids']
    contracts=[json.loads((d/'comparison_contract.json').read_text()) for d in (clean,e2e)]
    if any(c.get('schema')!='lm-paired-v1' for c in contracts) or contracts[0]['common']!=contracts[1]['common']:
        raise ValueError('paired common provenance differs')
    if [c['treatment'] for c in contracts]!=['diagnostic','e2e']:raise ValueError('wrong paired transcript treatments')
    rows=[checked_scores(d,set(ids)) for d in (clean,e2e)]
    reports=[json.loads((d/'merge_report.json').read_text()) for d in (clean,e2e)]
    complete=all(r['baseline_complete'] for r in reports) and all(set(r)==set(ids) and not any(v.get('pending') for v in r.values()) for r in rows)
    result={'comparison_complete':complete,'n':len(ids),'sources':[str(clean),str(e2e)],'common':contracts[0]['common']}
    if complete:
        per=[{'id':i,'diagnostic':rows[0][i]['score'],'e2e':rows[1][i]['score'],'delta':rows[1][i]['score']-rows[0][i]['score']} for i in ids]
        delta=[r['delta'] for r in per];n=len(delta)
        rng=random.Random(2026)
        boot=sorted(sum(rng.choices(delta,k=n))/n for _ in range(2000)) if n else []
        result.update(mean_delta=sum(delta)/n if n else None,delta_points=100*sum(delta)/n if n else None,
                      improved=sum(d>0 for d in delta),worsened=sum(d<0 for d in delta),unchanged=sum(d==0 for d in delta),
                      bootstrap_ci95=[100*boot[49],100*boot[1949]] if boot else None)
        atomic_write_text(clean.parent/'paired_per_id.jsonl',''.join(dumps(r)+'\n' for r in per))
    atomic_write_json(clean.parent/'paired_report.json',result)
    return result


In [ ]:
# Thư viện cài thêm: ưu tiên wheel từ output notebook `build-wheels` (Add Input), không có / lỗi thì cài từ mạng
WHEEL_DIRS = sorted({str(Path(p).parent) for p in glob.glob("/kaggle/input/**/wheels/*.whl", recursive=True)})
print("wheel dirs:", WHEEL_DIRS or "none -> install from internet")

def pip_install(pkgs: str) -> int:
    if WHEEL_DIRS:
        links = " ".join(f"--find-links {d}" for d in WHEEL_DIRS)
        if sh(f"{sys.executable} -m pip install -q --no-index {links} {pkgs}") == 0:
            print("installed from wheels:", pkgs)
            return 0
        print("wheel install failed -> internet:", pkgs)
    return sh(f"{sys.executable} -m pip install -q {pkgs}")

pip_install("sherpa-onnx soundfile")
# vLLM goes into its own venv: `pip install vllm` into the Kaggle env replaces torch (2.13+cu130 on the 2026-10 image)
# and breaks torchaudio -> transformers ASR. The base env stays untouched for ASR and the transformers fallback.
LLM_PYTHON = sys.executable
if CONFIG["llm"].get("backend", "auto") in ("auto", "vllm") and CONFIG.get("install_vllm", True):
    VENV = Path("/tmp/venv_vllm")  # not under /kaggle/working: it would be saved as output (GBs)
    pip_install("uv")
    sh(f"{sys.executable} -m uv venv -q {VENV} --python {sys.executable}")
    vpy = VENV / "bin" / "python"
    pkgs = f"{CONFIG.get('vllm_spec', 'vllm==0.32.0 wrapt')} "
    rc = 1
    if WHEEL_DIRS:
        links = " ".join(f"--find-links {d}" for d in WHEEL_DIRS)
        rc = sh(f"{sys.executable} -m uv pip install -q --python {vpy} --no-index {links} {pkgs}")
    if rc != 0:
        rc = sh(f"{sys.executable} -m uv pip install -q --python {vpy} {pkgs}")
    check = "import vllm, torch; assert torch.cuda.is_available(); print('vllm', vllm.__version__, 'torch', torch.__version__)"
    if rc == 0 and sh(f'{vpy} -c "{check}"') == 0:
        LLM_PYTHON = str(vpy)
print("LLM worker python:", LLM_PYTHON, "(transformers fallback)" if LLM_PYTHON == sys.executable else "(vLLM venv)")

In [ ]:
# Dataset: dùng bản đã Add Input trong /kaggle/input; nếu chưa có thì tự tải bằng kagglehub
# (https://www.kaggle.com/datasets/thanhkhil/zaloai-lifemate-dataset)
DATASET = "thanhkhil/zaloai-lifemate-dataset"
if not glob.glob(f"/kaggle/input/**/{CONFIG['split']}/{CONFIG['split']}.jsonl", recursive=True):
    pip_install("kagglehub")
    import kagglehub
    DATA_PATH = kagglehub.dataset_download(DATASET)
    print("dataset downloaded to", DATA_PATH)
    CONFIG["input_roots"] = [str(DATA_PATH)] + CONFIG["input_roots"]
else:
    print("dataset found under /kaggle/input")

In [ ]:
for m in list(sys.modules):
    if m == "lmlib" or m.startswith("lmlib."):
        del sys.modules[m]
from lmlib.common import env_info, find_split_dir, atomic_write_json, load_jsonl
info = env_info(); print(json.dumps(info, indent=1, ensure_ascii=False))
N_GPUS = len(info.get("gpus", []))
def split_dir(split):
    return find_split_dir(split, CONFIG["input_roots"], marker=f"{split}.jsonl")
atomic_write_json(OUT / "environment.json", info)

In [ ]:
# Resume: attach the output of an earlier version of this notebook as input and list its folder here; shard files
# of the same config are copied back (a different config hash is refused by the workers).
for d in CONFIG.get("resume_from_dirs", []):
    for p in glob.glob(f"{d}/**/*.shard*.jsonl", recursive=True):
        dst = OUT / Path(p).name
        if not dst.exists():
            shutil.copy(p, dst); print("resume <-", p)

In [ ]:
from lmlib.asr import launch as asr_launch, evaluate as asr_evaluate
from lmlib.data import load_rows, make_manifest

def run_asr(split, backend, subset="all", out_dir=None):
    sd = split_dir(split)
    rows = load_rows(sd)
    ids = None
    if subset != "all":
        man = make_manifest(load_rows(split_dir("train")), seed=CONFIG.get("manifest_seed", 2026))
        ids = sorted(set().union(*[man["ids"][s] for s in subset.split("+")]))
    a = dict(ASR_BACKENDS[backend])
    gpu = backend != "gipformer"
    n = max(N_GPUS, 1) if gpu else int(CONFIG.get("cpu_asr_workers", 2))
    cfg = {"split": split, "split_dir": str(sd), "asr": a, "subset_ids": ids, "num_shards": n, "out_dir": str(out_dir or OUT),
           "wallclock_s": CONFIG.get("wallclock_s", 10.5 * 3600), "cpu_threads": a.get("num_threads", 1)}
    p = Path(cfg["out_dir"]) / f"asr_{split}_{backend}.config.json"; atomic_write_json(p, cfg)
    t = time.time()
    codes = asr_launch(str(p), cfg, list(range(N_GPUS)) if gpu else [], str(LIB))
    print(split, backend, subset, "exit", codes, f"{(time.time() - t) / 60:.1f} min")
    for log in sorted(Path(cfg["out_dir"]).glob("asr_worker*.log")):
        print(log.read_text(encoding="utf-8")[-1500:])
    return cfg

In [ ]:
# Transcripts: attached ASR caches first, then ASR here for whatever is missing (not needed in diagnostic mode)
from lmlib.asr import load_cache
from lmlib.data import audio_paths
from lmlib.run import select_rows
SD = split_dir(CONFIG["split"])
rows = select_rows(CONFIG, load_rows(SD))
SFT_MODE = CONFIG["mode"] == "sft"  # needs transcripts of every train audio (ASR training variant)
ASR_SUBSET = "all" if (CONFIG["subset"] == "all" or SFT_MODE) else CONFIG["subset"]
if CONFIG["mode"] != "diagnostic":
    have = load_cache(CONFIG["asr_cache_dirs"], CONFIG["split"], CONFIG["asr_backend"])
    need = {p for r in (load_rows(SD) if SFT_MODE else rows) for p in audio_paths(r)}
    print(f"transcripts found: {len(need & set(have))}/{len(need)} audio ({CONFIG['asr_backend']})")
    if need - set(have):
        if not CONFIG["asr_inline"]:
            raise RuntimeError("missing transcripts: attach the lifemate_asr output or set asr_inline=True")
        run_asr(CONFIG["split"], CONFIG["asr_backend"], ASR_SUBSET)
        have = load_cache(CONFIG["asr_cache_dirs"], CONFIG["split"], CONFIG["asr_backend"])
        print(f"after ASR: {len(need & set(have))}/{len(need)}")

## SFT LoRA (chỉ mode `sft`): một tiến trình train mỗi GPU, mỗi biến thể một GPU

In [ ]:
if CONFIG["mode"] == "sft" or CONFIG["llm"].get("adapter"):
    pip_install("peft")
from lmlib.agent import Catalog
from lmlib.asr import load_cache
from lmlib.sft import build_examples, merge_latest

def merge_adapter_from(ckpt_dir, name):
    out = Path("/tmp/merged") / name  # merged weights stay out of /kaggle/working (GBs)
    if not (out / "config.json").exists():
        merge_latest(CONFIG["llm"]["model"], ckpt_dir, out, CONFIG["llm"].get("revision"))
    return out

MERGED = {}
if CONFIG["mode"] == "sft":
    all_rows = load_rows(SD)
    fit = select_rows(dict(CONFIG, subset=SFT["train_subset"], limit=None), all_rows)
    catalog = Catalog(json.loads((SD / f"{SD.name}_tools_catalog.json").read_text(encoding="utf-8")))
    asr_texts = load_cache(CONFIG["asr_cache_dirs"], CONFIG["split"], CONFIG["asr_backend"])
    variants = SFT["variants"][: max(N_GPUS, 1)]
    procs = []
    for gi, v in enumerate(variants):
        ex = build_examples(fit, catalog, CONFIG["agent"]["result_chars"], texts=asr_texts if v == "asr" else None)
        vdir = OUT / f"sft_{v}"; vdir.mkdir(parents=True, exist_ok=True)
        ex_path = vdir / "examples.jsonl"
        ex_path.write_text("".join(json.dumps(e, ensure_ascii=False) + "\n" for e in ex), encoding="utf-8")
        tcfg = {k: v_ for k, v_ in SFT.items() if k not in ("train_subset", "variants")}
        tcfg.update(model=CONFIG["llm"]["model"], revision=CONFIG["llm"].get("revision"), out_dir=str(vdir),
                    examples_path=str(ex_path), resume_dirs=list(CONFIG.get("resume_from_dirs", [])), variant=v)
        cp = vdir / "train_config.json"; atomic_write_json(cp, tcfg)
        print(v, "examples:", len(ex), "from", len(fit), "train_fit samples")
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gi), PYTHONUNBUFFERED="1", PYTHONPATH=str(LIB) + os.pathsep + os.environ.get("PYTHONPATH", ""))
        log = open(vdir / "train.log", "a", encoding="utf-8")
        procs.append((v, subprocess.Popen([sys.executable, "-m", "lmlib.sft", "--config", str(cp)], env=env, stdout=log, stderr=subprocess.STDOUT), log))
    t0 = time.time()
    for v, p, log in procs:
        print(v, "train exit", p.wait(), f"{(time.time() - t0) / 60:.1f} min"); log.close()
        print((OUT / f"sft_{v}" / "train.log").read_text(encoding="utf-8")[-2500:])
    for v, _, _ in procs:
        try:
            MERGED[v] = merge_adapter_from(OUT / f"sft_{v}", v)
        except Exception as e:
            print("merge failed", v, type(e).__name__, e)
    print("merged:", MERGED)
elif CONFIG["llm"].get("adapter"):
    CONFIG["llm"]["model"] = str(merge_adapter_from(CONFIG["llm"]["adapter"], "adapter"))
    print("agent model <- merged adapter", CONFIG["llm"]["model"])

## Chạy agent (một worker / GPU, ghi bền từng mẫu)

In [ ]:
from lmlib.run import launch, merge

def run_agent(conf, out_dir):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    cfg = dict(conf, split_dir=str(SD), num_shards=max(N_GPUS, 1), out_dir=str(out_dir))
    cfg_path = out_dir / "config.json"; atomic_write_json(cfg_path, cfg)
    t0 = time.time()
    codes = launch(str(cfg_path), cfg, list(range(N_GPUS)), str(LIB), python=LLM_PYTHON)
    print("exit codes:", codes, f"{(time.time() - t0) / 60:.1f} min")
    for log in sorted(out_dir.glob("run_worker*.log")):
        print("=====", log.name); print(log.read_text(encoding="utf-8")[-2500:])
    rep = merge(cfg)
    print(json.dumps(rep, ensure_ascii=False, indent=1))
    return cfg, rep

if CONFIG["mode"] == "sft":
    reports = {}
    for v, model_dir in MERGED.items():
        conf = dict(CONFIG, mode="e2e", experiment=f"{CONFIG['experiment']}-{v}", llm=dict(CONFIG["llm"], model=str(model_dir)))
        reports[v] = run_agent(conf, OUT / f"eval_{v}")[1]
    atomic_write_json(OUT / "sft_eval_summary.json",
                      {v: {k: r.get(k) for k in ("n", "completed", "score", "hash_ok_rate", "by_group", "finished_counts")} for v, r in reports.items()})
    cfg, rep = None, {}
else:
    cfg, rep = run_agent(CONFIG, OUT)

In [ ]:
if rep.get("submission") and CONFIG["mode"] == "public":
    shutil.copy(rep["submission"], WORK / "submission.jsonl")
    # the CLI runs from inside the split folder with --root .. (dataset README)
    sh(f"cd {SD} && {sys.executable} -m zaic_tools --split {SD.name} --root .. --validate {WORK / 'submission.jsonl'} | tail -3")
for p in sorted(OUT.rglob("summary.shard*.json")):
    s = json.loads(p.read_text()); print(p.relative_to(OUT), s["s_per_sample"], "s/sample (batched, not the A30 SLA)")

In [ ]:
# A few traces to eyeball
tr = [json.loads(l) for p in sorted(OUT.rglob("traces.shard*.jsonl")) for l in open(p, encoding="utf-8").read().splitlines()[1:]]
for t in tr[:5]:
    print(t["id"], "|", t["request"], "| finished:", t["finished"])
    for s in t["steps"]: print("   ", json.dumps({k: s[k] for k in s if k != "result"}, ensure_ascii=False)[:300])